Politecnico di Torino\
Corso di Laurea in Matematica per l'Ingegneria\
Codice commentato per la tesina di Matematica per l'Intelligenza Artificiale, A.A. 2025/2026\
**Studente**: Francesca Gregorio\
**Matricola**: s322444

# Analisi del Dataset MAGIC Gamma Telescope

# Il Dataset e l' EDA (Explorative Data Analysis)



Il [Dataset \"MAGIC Gamma
Telescope\"](https://archive.ics.uci.edu/dataset/159/magic+gamma+telescope)
è un insieme di $N = 19020$ osservazioni
generate tramite simulazione Monte Carlo e rappresenta eventi registrati
da un telescopio Cherenkov a terra. Ciascuna informazione descrive le
caratteristiche geometriche e fotometriche dell'ellisse prodotta
dall'immagine di uno sciame atmosferico e appartiene a due classi:
*gamma (g)*, ovvero il segnale di interesse e *hadron (h)*, ovvero il
rumore di fondo cosmico.\
Si tratta, dunque, di un problema di classificazione binaria
supervisionata, il cui obiettivo è distinguere eventi gamma da eventi
andronici a partire da dieci feature continue, i *parametri di Hillas*.
Data la natura del problema, si adotterà come metrica principale la
curva ROC e l'AUC, come indicato esplicitamente nella documentazione
ufficiale del dataset: classificare
erroneamente un evento di fondo è più penalizzante dell'errore opposto.
## Importazione del dataset e prima esplorazione
Per prima cosa, vengono importate le librerie che torneranno utili in seguito. Tra esse particolare rilievo rivesono sklearn, pandas, numpy e matplotlib

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import sklearn
from sklearn.preprocessing import LabelEncoder
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis as LDA
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis as QDA
from FisherDA import MultipleFisherDiscriminantAnalysis as MDA
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.svm import SVC, LinearSVC
from mpl_toolkits.mplot3d import Axes3D
from IPython.display import display
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import roc_auc_score, roc_curve, ConfusionMatrixDisplay, confusion_matrix, accuracy_score, precision_score, recall_score
import itertools
import plotly
import plotly.express as px #plotly serve per visualizzazione interattiva
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings("ignore")
print("Librerie caricate correttamente")

## Importazione del dataset
Si importa il dataset e si preparano le osservazioni per le successive analisi statistiche e applicazioni di machine learning.<df> L'obiettivo è ottenere una struttura dei dati chiara e coerente, verificando la qualità delle informazioni disponibili e predisponendo correttamente le variabili per l'applicazione di tecniche di classificazione e riduzione della dimensionalità.

In [ ]:
df = pd.read_csv("magic04.data", header=None)
df.head()

Si osserva che il file **NON** ha un header. Inoltre, le colonne verranno assegnate successivamente.<br>
In seguito, viene stampata la descrizione del dataset, tratta dalla documentazione ufficiale del repository UCI Machine Learning Repository, in modo da fornire un contesto chiaro sulle caratteristiche dei dati e sul problema di classificazione che si sta per affrontare.

In [ ]:
with open("magic04.names", "r", encoding = "utf-8", errors = "ignore") as f:
    text = f.read()
print(text)

Gli attributi del dataset sono:
- *fLength*, float64, lunghezza dell'asse maggiore dell'ellisse (mm);
- *fWidth*, float64, lunghezza dell'asse minore dell'ellisse (mm);
- *fSize*, float64, logaritmo in base 10 della somma del contenuto di tutti i pixel (unità di misura: numero di fotoni);
- *fConc*, float64, rapporto tra la somma dei due pixel più luminosi e *fSize*;
- *fConc1*, float64, rapporto tra il pixel più luminoso e *fSize*;
- *fAsym*, float64, distanza dal pixel più luminoso al centro, priettata sull'asse maggiore (mm);
- *fM3Long*, float64, radice cubica del momento terzo lungo l'asse maggiore (mm); indica l'asimmetria o "skewness" della distribuzione lungo l'asse;
- *fM3Trans*, float64, radice cubica del momento terzo lungo l'asse minore (mm);
- *fAlpha*, float64, angolo tra l'asse maggiore e il vettore che punta all'origine (gradi);
- *fDist*, float64, distanza dall'origine al centro dell'ellisse (mm).

Alla luce di queste informazioni, vengono assegnati dei nomi significativi alle colonne per facilitare l'interpretazione delle caratteristiche fisiche degli eventi registrati dal telescopio.

In [ ]:
columns = ["fLength", "fWidth", "fSize", "fConc", "fConc1", "fAsym", "fM3Long", "fM3Trans", "fAlpha", "fDist", "class"]
df.columns = columns
df.head()

In [ ]:
df.tail()

Si stampano, ora, le informazioni principali del dataset. In particolare, già dalla visualizzazione appena effettuata delle prime e ultime righe, si nota che i dati registrati in *class* appartengono a un tipo diverso rispetto agli altri. Inoltre, le feature esplicative presentano valori con scale molto diverse tra loro, suggerendo la necessità di standarizzare i dati prima dell'applicazione degli algoritmi.

In [ ]:
df.info()

Si osserva che le dieci feature esplicative sono di tipo *float64*, coerentemente con la natura fisica delle misurazioni (distanze in mm, angoli in gradi, rapporti adimensionali). <br> La variabile "*class*" è di tipo *object*. <br> Dal momento che il dataset ha 19020 osservzioni e per ogni feature si hanno 19020 valori non nulli, si conclude che il dataset MAGIC Gamma Telescope non contiene valori nulli.

Per comprenderne la dimensionalità e la struttura generale, si stampano le dimensioni del datase.

In [ ]:
print("Numero di osservazioni:", df.shape[0])
print("Numero di feature:", df.shape[1]-1)

Il dataset è composto da 19020 osservazioni e 11 colonne, divise tra 10 feature numeriche continue e una variabile target "*class*". Il dataset può, dunque, essere scomposto in una matrice
$\mathbf{X} \in \mathbb{R}^{N \times 10}$, in cui ogni riga corrisponde
a un elemento del dataset e ogni colonna a un attributo, e un vettore
colonna $\mathbf{y} \in \{0,1\}^N$, contentente la classe corrispondente
a ciascuna riga di $\mathbf{X}$, codificata come un intero pari a $0$
(corrispondente a *gamma, g*) o a $1$ (corrispondente a *hadron, h*),
con la tecnica del **Label Encoding** che preserva l'informazione categoriale necessaria per il processo di classificazione (binaria). Questo passaggio è necessario poichè gli algoritmi di machine learning operano su dati numerici e non sono in grado di gestire direttamente etichette testuali. Dal momento che tutte le variabili esplicative del dataset sono di tipo numerico, esse rimangono invariate rispetto al dataset originale e non occorre usare il One-Hot Encoding.
<br> La dimensione del dataset è sufficientemente ampia da consentire una suddivisione robusta in training, validation e test set. <br> Viene, inoltre, creata una copia del dataset con la quale effettuare le successive analisi e trasformazioni.

In [ ]:
magic = df.copy()
le = LabelEncoder()
magic.loc[:, "class"] = le.fit_transform(magic["class"])
le.classes_
print("Mappatura classi con LabelEncoder:")
for index, class_name in enumerate(le.classes_):
    print(f"Classe {index}: {class_name}")
magic.head()

Per valutare l'eventuale presenza di sbilanciamenti, che potrebbbero influenzare le prestazioni dei modelli di classificazione, si verifica la distribuzione all'interno delle classi.

In [ ]:
abs_f = df["class"].value_counts()
rel_f = df["class"].value_counts(normalize=True)
freq_table = pd.DataFrame({"Absolute Frequency": abs_f, "Relative Frequency": rel_f})
freq_table

Le classi del dataset risultano moderatamente sbilanciate, con una prevalenza della classe *g* (circa il 64.8% delle osservazioni). Tale distribuzione, tuttavia, è sufficientemente equilibrata da non richiedere tecniche di ricampionamento, ma viene comunque tenuta in considerazione nella valutazione della prestazione dei modelli. <br>Inoltre, come indicato nella documentazione ufficiale al dataset, tale sbilanciamento non riflette la realtà fisica, dove gli eventi adronici costituiscono la maggioranza. <br> Tale asimmetria rende l'accuracy una metrica poco affidabile e, per questo motivo, la prestazione dei modelli sarà valutata con la curva ROC (*Receiver Operating Characteristic*) e l'AUC (*Area Under the Curve*).

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12,4))
#frequenza assoluta
ax[0].bar(abs_f.index, abs_f.values)
ax[0].set_xlabel("Class")
ax[0].set_ylabel("Absolute Frequency")
ax[0].set_title("Class Distribution")
# frequenza relativa
rel_f_perc = rel_f * 100
ax[1].bar(rel_f_perc.index, rel_f_perc.values)
ax[1].set_xlabel("Class")
ax[1].set_ylabel("Relative Frequency (%)")
ax[1].set_title("Relative Class Distribution")
plt.tight_layout()
plt.show()

Prima di procedere, si analizzano le principali statistiche descrittive delle variabili, al fine di comprendere la distribuzione dei dati e individuare eventuali anomalie o scale differenti tra le feature.

In [ ]:
magic.describe()

In [ ]:
magic.drop("class", axis=1).var().sort_values(ascending=False)

Si osserva una forte eterogeneità nelle varianze delle variabili. Tra la feature più dispersa (*fDist*) e quella meno dispersa (*fConc1*) il rapporto tra le varianze è superiore a 450000. Questa differenza di scala renderebbe qualsiasi algoritmo basato su distanze o proiezioni dominato dalle variabili a varianza maggiore. Si osserva che non sono presenti feature a varianza nulla che non darebbero alcun contributo nella PCA. <br> Prima di procedere all'applicazione della PCA e dell'SVM occorre, dunque, attuare un processo di standarizzazione (*si veda in seguito*).

Si analizza la matrice di correlazione tra le variabili per individuare eventuali dipendenze lineari. La presenza di correlazioni elevate suggerisce che una riduzione della dimensionalità tramite PCA possa essere efficace. Feature altamente scorrelate, invece, indicano che una "compressione" porterebbe a una perdita significativa di potere discriminante.

In [ ]:
corr = magic.drop("class", axis=1).corr()
plt.figure(figsize=(4,4))
plt.matshow(corr, fignum=1)
plt.colorbar()
plt.xticks(range(len(corr.columns)), corr.columns, rotation = 45, ha = "left")
plt.yticks(range(len(corr.columns)), corr.columns)
plt.title("Correlation Matrix", pad=20)
plt.show()

La matrice di correlazione mostra alcune correlazioni moderate tra le feature geometrice dell'ellisse. In particolare *fLenght* e *fWidth* mostrano una certa correlazione positiva. Inoltre, *fConc* e *fConc1* che misurano grandezze simili (concentrazione dei pixel più luminosi) presentano una correlazione molto prossima a 1.00. <br>
D'altra parte, feature come *fAlpha*, *fAsym* e *fDist* presentano una correlazione quasi nulla con il resto dei parametri.
Questo comportamento indica che l'orientamento, l'asimmetria e la posizione geometrica dello sciame forniscono informazioni fisiche uniche e complementari, non comprimibili senza una significativa perdita di potere discriminante. La presenza di queste dimensioni indipendenti giustifica la complessità intrinseca del problema e spiega perché, in fase di riduzione dimensionale (PCA), sia necessario mantenere un numero relativamente elevato di componenti principali per preservare la varianza totale e garantire l'accuratezza dei successivi modelli di classificazione.

Realizzazione di istogrammi per visualizzare la distribuzione delle variabili, eventuali asimmetrie e differenze di scala tra le feature:

In [ ]:
magic.drop("class", axis=1).hist(figsize=(12,8), bins=20)
plt.tight_layout()
plt.show()

Le distribuzioni delle features sono asimmetriche e non gaussiane. *fAlpha* mostra una distribuzione quasi uniforme sull'intervallo [0°, 90°], come ci si aspetta dagli eventi di fondo adronico, mentre per gli eventi gamma ci si aspetta una concentrazione verso 0°; questo comportamento fisico si riflette nell'istogramma. *fAsym*, *FM3Long* e *FM3Trans* mostrano distribuzioni centrate in zero e con code pesanti, indicando la presenza di eventi con asimmetrie pronunciate.

Per analizzare la distribuzione delle variabili nelle due classi del dataset, ci si serve di boxplot. Essi, infatti, permettono di evidenziare differenze nelle mediane, nella dispersione dei dati e nella presenza di outlier, fornendo informazioni preliminari sulla capacità discriminante delle feature.

In [ ]:
fig, axs = plt.subplots(5,2, figsize=(12,14))
axs = axs.ravel()
features = magic.drop("class", axis=1).columns
for i, feature in enumerate(features):
    
    magic.boxplot(
        column=feature,
        by="class",
        ax=axs[i]
    )
    
    axs[i].set_title(feature)
    axs[i].set_xlabel("Class")
    axs[i].set_ylabel(feature)

plt.suptitle("Feature Distribution by Class")

plt.tight_layout()

plt.show()

I boxplot confermano la presenza di forti outlier in quasi tutte le features, in particolare per quanto riguarda *fAsym*, *fM3Long*, *fDist* e *fLength*. Fisicamente, tali valori estremi rappresentano eventi rari, legati a sciami particolarmente energetici o geometrie inusiuali. Essi possono rivelarsi discriminanti ai fini della classificazione.

## Suddivisione del dataset e standarizzazione
Le variabili esplicative del dataset \(**X**\) vengono separate dalla variabile target \(*y*, classi da predire\). Tale distinzione è di vitale importanza nei modelli di apprendimento supervisionato, in cui l'obiettivo è apprendere una funzione che, a partire dalle feature, consenta di prevedere correttamente la classe associata a ciascuna osservazione. <br>
Successivamente, i dati i dati in training (per l'addestramento dei modelli), validation (per valutare gli iperparametri dei modelli al fine di scegliere i migliori) e test set (per valutare i risultati), preservando la proporzione tra classi e tenendo separati i dati utilizzati per l'addestramento da quelli impiegati per la valutazione del modello. Infine, viene applicata una standarizzazione delle feature affinchè variabili caratterizzate da scale diverse diventino confrontabili. L'applicazione (e in particolare fitting) della standarizzazione è tale per cui si evita il data leakage.

In [ ]:
#separazione feature-target
X = magic.drop("class", axis=1)
y = magic["class"].astype(int) #tornerà utile nei calcoli successivi
#suddivisone train+validation - test set
random_state = 42
t_size = 0.35
X_trainval, X_test, y_trainval, y_test = train_test_split(X, y, test_size = t_size, stratify = y, random_state= random_state, shuffle = True) #stratified per mantenere proporzioni tra classi
#suddivisione train-validation set
v_size = 15/65 #test --> 0.35, val --> 0.15, tr --> 0.5
X_train, X_val, y_train, y_val = train_test_split(X_trainval, y_trainval, test_size = v_size, stratify = y_trainval, random_state=random_state, shuffle=True)
#standarizzazione
scaler = StandardScaler()
X_trainval_scaled_tv = scaler.fit_transform(X_trainval) #fitting per quando si usa train+val come training set
X_test_scaled_tv = scaler.transform(X_test)
X_val_scaled_tv = scaler.transform(X_val)
X_train_scaled_tv = scaler.transform(X_train)
X_scaled_tv = scaler.transform(X)
scaler2 = StandardScaler()
X_train_scaled = scaler2.fit_transform(X_train) #fitting per quando si effettua grid search e quindi si utilizza il validation set come tale
X_val_scaled = scaler2.transform(X_val)
X_trainval_scaled = scaler2.transform(X_trainval)
X_test_scaled = scaler2.transform(X_test)
X_scaled = scaler2.transform(X)
#stampo informazioni
def class_balance(yy):
    return f"{(yy==0).mean()*100:.1f}% gamma / {(yy==1).mean()*100:.1f}% hadron"
M = X.shape[0]
split_summary = pd.DataFrame({
    "Insieme":    ["Training", "Validation", "Test", "Training+Val"],
    "Cardinalità":[len(y_train), len(y_val), len(y_test), len(y_trainval)],
    "Frazione":   [f"{len(y_train)/M*100:.1f}%", f"{len(y_val)/M*100:.1f}%",
                   f"{len(y_test)/M*100:.1f}%", f"{len(y_trainval)/M*100:.1f}%"],
    "Bilanciamento classi": [class_balance(y_train), class_balance(y_val),
                              class_balance(y_test), class_balance(y_trainval)],
})
split_summary

Il dataset è stato suddiviso in tre parti secondo uno schema 50/15/35: training set (50%, 9510 campioni), validation set (15%, 2853) e test set (35%, 6657). In questo modo, sono garantiti un validation e un test set statisticamente robusti. Per garantire casualità e riproducibilità, la suddivisione è stata effettuata con shuffle = True e random_state = 42 (e tale seed sarà utilizzato anche in seguito). <br> 

# Riduzione della dimensionalità: PCA



Ogni elemento $x$ del dataset può essere visto come un vettore di
$\mathbb{R}^{n}$, dove $n$ è il numero di feature. Dal momento che l'alta dimensionalità potrebbe causare dei
problemi nel learning, è lecito domandarsi se sia possibile rappresentare
i punti in dimensioni più basse senza perdere l'informazione in essi
contenuta.\
Nel caso in esame il numero di feature non è eccessivamente elevato, ma si è scelto comunque di effettuare la riduzione
di dimensionalità per scopi didattici per confrontare le tecniche della
*Principal Component Analysis (PCA)* e della *Fisher Discriminant
Analysis (FDA)*. Nella maggior parte dei casi, tuttavia, la classificazione sarà applicata alle
feature standarizzate ma di dimensione non ridotta. Questa scelta è
dettata dal fatto che i *parametri di Hillas*, ovvero le 10 feature del
dataset, sono state progettate appossitamente per discriminare la classe
gamma da quella adronica.



## Principal Component Analysis (PCA)

La *Principal Component Analysis (PCA)* è una tecnica di riduzione della
dimensionalità ampiamente utilizzata nell'ambito del machine learning e
della data analysis. Dato un insieme di osservazioni caratterizzate da
un elevato numero di variabili, la PCA consente di rappresentare i dati
mediante un numero ridotto di feature, dette *componenti principali*
(*Principal Components, PC*), preservando il più possibile
l'informazione contenuta nel dataset originale. L'idea alla base della
PCA consiste nell'individuare nuove direzioni dello spazio delle feature
lungo le quali i dati presentano la massima varianza. Le componenti
principali vengono costruite imponendo che siano mutuamente ortogonali:
la prima individua la direzione di massima varianza, la seconda cattura
la massima varianza residua soggetta al vincolo di ortogonalità rispetto
alla prima, e così via\... Le PC risultano, quindi, scorrelate e
ordinate secondo la quantità di varianza spiegata.\
Oltre a ridurre la dimensionalità del problema, la PCA permette di
eliminare parte del rumore presente nei dati, scartando le componenti
associate ad autovalori molto piccoli e mitigando così gli effetti del
*curse of dimensionality* e facilitando la visualizzazione di dataset
complessi in due o tre dimensioni. Sia dato un training set di $m$
osservazioni
$\chi = \{x^{(1)}, x^{(2)}, \dots, x^{(m)}\} \subset \mathbb{R}^n,$
dove $n$ rappresenta il numero di feature osservate. Si definisce il
vettore delle medie campionarie
$$
\mu = \frac{1}{m}\sum_{i=1}^{m}x^{(i)}.
$$ 
Il primo passo consiste nel
centrare i dati, imponendo che ciascuna feature abbia media nulla. Si
ottiene così la matrice centrata 
$$
C \in \mathbb{R}^{m \times n},
$$ 
i
cui elementi sono definiti da 
$$
C_{ij} = x_j^{(i)} - \mu_j,
\qquad
i = 1,\dots,m,
\qquad
j = 1,\dots,n.
$$
Il centraggio è fondamentale poiché garantisce che
l'analisi della varianza venga effettuata rispetto al centro dei dati e
non rispetto all'origine del sistema di riferimento. Le relazioni di
covarianza tra le variabili sono descritte dalla matrice
$$
S = \frac{1}{m-1}C^\top C,
$$ 
detta *matrice di covarianza*. Per
costruzione, $S$ è simmetrica e semidefinita positiva; di conseguenza,
ammette una base ortonormale di autovettori ed è diagonalizzabile.\
Fissata una dimensione ridotta $d < n$, l'obiettivo della PCA consiste
nel trovare il sottospazio di dimensione $d$ che minimizza la perdita di
informazione dovuta alla riduzione dimensionale. Sia $U$ in
$\mathbb{R}^{n \times d}$ la matrice di proiezione sul sottospazio di
dimensionalità $d$. La rappresentazione del campione $x^{(i)}$ nello
spazio ridotto è ottenuta mediante
$$
z^{(i)} = U^\top x^{(i)} \in \mathbb{R}^d.
$$ 
A partire da tale
rappresentazione, è possibile ricostruire un'approssimazione del dato
originale tramite 
$$
\hat{x}^{(i)}
=
Uz^{(i)}
=
UU^\top x^{(i)}.
$$ 
L'errore di ricostruzione associato al campione è
quindi 
$$
\left\|
x^{(i)}-\hat{x}^{(i)}
\right\|_2^2.
$$ 
Il problema può essere formulato come la minimizzazione
dell'errore di ricostruzione: 
$$
\arg\min_{U \in \mathbb{R}^{n \times d}}
\sum_{i=1}^{m}
\left\|
x^{(i)} - UU^\top x^{(i)}
\right\|_2^2,
$$
dove le colonne
di $U$ costituiscono una base ortonormale del sottospazio cercato. <br> Equivalentemente, si vuole determinare un insieme ortonormale
$$
\{u_1,\dots,u_d\}
$$ tale che, detta 
$$
\Pi : \mathbb{R}^n \to
\mathrm{span}\{u_1,\dots,u_d\},
$$
 la proiezione ortogonale sul
sottospazio generato da tali vettori minimizzi la quantità 
$$
D_S =
\sum_{i=1}^{m}
\left\|
x^{(i)}-\Pi(x^{(i)})
\right\|_2^2.
$$ 
La
soluzione è fornita dagli autovettori della matrice di covarianza
associati agli autovalori maggiori. Indicando con
$$
\lambda_1 \geq \lambda_2 \geq \dots \geq \lambda_n
$$
gli autovalori di
$S$, che misurano la varianza dei dati spiegata lungo la direzione
individuata dall'autovettore corrispondente. Le prime $d$ componenti
principali sono, quindi, date dagli autovettori
$$
pc_1, pc_2, \dots, pc_d.
$$ 
La matrice di proiezione risulta, quindi:
$$
U_d =
\begin{bmatrix}
pc_1, & pc_2, & \dots & pc_d
\end{bmatrix}.
$$ 
Risulta, inoltre, che lo spazio di proiezione è 
$$
span\{pc_1, pc_2, \dots, pc_d\}.
$$
È lecito domandarsi come scegliere il valore ottimale
di $d$ in modo da ridurre la perdita di informazione. A tale scopo, si
introduce la *proporzione di varianza spiegata* (*Proportion of Variance
Explained, pve*).\
La proporzione di varianza spiegata dalla $i$-esima componente
principale è definita come 
$$
pve_i =
\frac{\lambda_i}{\sum_{j=1}^{n}\lambda_j} = \frac{\sigma_i^2}{\sum_{j = 1}^{n}\sigma_j^2},
\qquad i = 1,\dots,n.
$$ 
La quantità 
$$
\mathrm{cumulative}\; pve_k
=
\sum_{i=1}^{k} pve_i
=
\frac{\sum_{i=1}^{k}\lambda_i}{\sum_{j=1}^{n}\lambda_j}
$$ 
rappresenta
invece la *proporzione cumulativa di varianza spiegata* dalle prime $k$
componenti principali.\
Fissata una soglia di varianza cumulativa spiegata, tipicamente compresa
tra il $90\%$ e il $95\%$, si sceglie il più piccolo valore di $d$ tale
che $\mathrm{cumulative}\; pve_d$ superi la soglia prefissata.

### Un'alternativa al centraggio dei dati: la standarizzazione

Analogamente alla media, si definisce la deviazione standard campionaria associata
alla $j$-esima variabile:
$$
\sigma_j =
\sqrt{
\frac{1}{m-1}
\sum_{i=1}^{m}
\left(x_j^{(i)}-\mu_j\right)^2
},
\qquad j = 1,\dots,n.
$$
Effettuando la standarizzazione dei dati, ovvero
imponendo che ciascuna feature abbia media nulla e varianza unitaria si
ottiene la matrice $\tilde C \in \mathbb{R}^{m\times n}$, definita da
$$
\tilde C_{ij} = \frac{x_j^{(i)}-\mu_j}{\sigma_j},
\quad
i=1,\dots,m,
\quad
j=1,\dots,n.
$$ Le informazioni sulla correlazione dei dati sono
contenute nella matrice $\tilde S =
\frac{1}{m-1}\tilde C^\top \tilde C.$ Essa, di fatto, coincide con la
*matrice di correlazione*. Infatti, la matrice di correlazione si definisce $R = (\rho_{ij})_{i,j = 1}^n$, dove ciascun elemento $\rho_{ij}$ rappresenta il *coefficiente di correlazione lineare di Pearson*. Due feature fortemente correlate sono
segnale di ridondanza informativa e ciò giustifica la riduzione della
dimensionalità effettuata tramite la PCA.

In generale la PCA può essere applicata sia alla matrice di covarianza
sia alla matrice di correlazione dei dati. La scelta tra i due approcci
dipende principalmente dalle caratteristiche delle variabili del
dataset.\
La matrice di covarianza conserva l'informazione relativa alla
variabilità assoluta delle feature. Tuttavia, quando le variabili
presentano scale numeriche molto diverse oppure sono espresse in unità
di misura differenti, le feature caratterizzate da varianza più elevate
tendono a dominare l'analisi. In queste situazioni, le componenti
principali risultano fortemente influenzate dalle variabili
numericamente predominanti, pur non essendo necessariamente quelle più
informative dal punto di vista strutturale.\
La standardizzazione rende, inoltre, le feature confrontabili tra loro,
permettendo alla PCA di riflettere le relazioni strutturali presenti tra
le feature e migliorando l'interpretabilità dei risultati.\
Si consideri, ad esempio, un dataset contenente:

- l'età di un individuo (in anni);

- il reddito medio annuo (in Euro);

- l'altezza in (cm);

- altre feature ...

Senza effettuare la standarizzazione, la variabile relativa al reddito
presenta una varianza molto più elevata rispetto alle altre feature e
dominerebbe la costruzione delle componenti principali. Prima di
procedere all'analisi è opportuno, quindi, riportare tutte le variabili
sulla stessa scala affinchè possano contribuire equamente alla
costruzione delle PC.<br> Se si considera, invece, un
dataset contenente:

- la temperatura minima giornaliera (in gradi Celsius);

- la temperatura massima giornaliera (in gradi Celsius);

- la temperatura media giornaliera (in gradi Celsius).

In questo caso, i dati presentano ordini di grandezza simili e stessa
unità di misura, quindi non è limitante effettuare il centraggio dei
dati.

## PCA e il MAGIC Gamma Telescope

Si applica la PCA (Principal Component Analysis) per ridurre la dimensionalità del dataset, preservando la maggior parte della varianza informativa. La PCA è particolarmente indicata in presenza di feature correlate e con scale differenti; entrambe le condizioni sono verificate nell'analisi precedente. <br>
La trasformazione viene appresa sul training+validation set standarizzato e successivamente applicata al test set (standarizzato), per evitare data leakage*. <br>
<br>
<small> *In machine learning, per data leakage si intende l'errore che si verifica quando informazioni provenienti dal test set "trapelano" nel set di addestramento </small>

Per prima cosa, si esegue la PCA conservando tutte le 10 componenti principali, al fine di analizzare la varianza spiegata da ciascuna e scegliere il numero ottimale di componenti da mantenere. Inoltre, si confronta la varianza spiegata dalla prima componente principale sui dati grezzi con quella calcolata sui dati standarizzati per ribadire l'importanza del processo di scaling nel caso in esame.

In [ ]:
#inizializzazione dell'oggetto PCA
pca_global = PCA()
pca_grezza = PCA()
#"Fit" dell'oggetto PCA
pca_grezza.fit(X_trainval)
pca_global.fit(X_trainval_scaled_tv)
print(f"Varianza spiegata dalle prime due PC- dati non standarizzati: {(pca_grezza.explained_variance_ratio_[0]+pca_grezza.explained_variance_ratio_[1])*100:5.1f}")
print(f"Varianza spiegata dalle prime due PC - dati standarizzati: {(pca_global.explained_variance_ratio_[0]+ pca_global.explained_variance_ratio_[1])*100:5.1f}")
#varianza spiegata
expl_var_global = pca_global.explained_variance_ratio_
cum_var_global = np.cumsum(expl_var_global) #varianza cumulata


Sui dati non standarizzati, le prime due PC catturano una frazione enorme molto maggiore della varianza, in quanto tendono ad allinearsi con le feature con scale maggiori. Per questo motivo, per il resto dell'analisi si utilizzeranno i dati standarizzati.

In [ ]:
print("PC & varianza spiegata sui dati riscalati - train+val:")
#print
for i, (ev, cv) in enumerate(zip(expl_var_global, cum_var_global)):
    print(f"PC{i+1:2d} - varianza spiegata: {ev:.4f} | cumulata: {cv:.4f}")

Visualizzazione dell'andamento della varianza spiegata cumulativa all'aumentare del numero di componenti principali considerate sul training + validation:

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
# Scree plot
axes[0].bar(range(1, pca_global.n_features_in_ +1), expl_var_global, color='steelblue', edgecolor='black')
axes[0].set_xlabel("Componente principale")
axes[0].set_ylabel("Varianza spiegata")
axes[0].set_title("MAGIC Gamma Telescope - PCA_global - Varianza spiegata")
axes[0].set_xticks(ticks=np.arange(1, pca_global.n_features_in_ + 1),
           labels=[f'PC{i}' for i in range(1, pca_global.n_features_in_ + 1)],
           rotation=30)
# Varianza cumulata
axes[1].plot(range(1, pca_global.n_features_in_ + 1), cum_var_global, marker='o', color='steelblue')
axes[1].axhline(y=0.90, color='red', linestyle='--', label='90% varianza spiegata')
axes[1].axhline(y=0.95, color='orange', linestyle='--', label='95% varianza spiegata')
axes[1].set_xlabel("Numero di componenti")
axes[1].set_ylabel("Varianza spiegata cumulata")
axes[1].set_title("MAGIC Gamma Telescope - Varianza Cumulata")
axes[1].set_xticks(ticks=np.arange(1, pca_global.n_features_in_ + 1),
           labels=[f'PC{i}' for i in range(1, pca_global.n_features_in_ + 1)],
           rotation=30)
axes[1].legend()
plt.tight_layout()
plt.show()
k_95 = int(np.searchsorted(cum_var_global, 0.95)+1)
k_90 = int(np.searchsorted(cum_var_global, 0.90)+1)
print(f"Servono {k_90} componenti principali per il 90% della varianza spiegata")
print(f"Servono {k_95} componenti principali per il 95% della varianza spiegata")

Lo scree plot (istogramma sulla sinistra) mostra un contibuto dominante da parte della PC1, con una varianza spiegata pari al 42.29%. Esso è seguito da un decremento progressivo e senza un vero e proprio "gomito" netto: la PC2 spiega il 15,75% della varianza, la PC3 il 10.14%, mentre la PC4 il 9.79%. <br>
Ciò significa che l'informazione non è concentrata sulle prime  componenti, ma è distribuita in maniera uniforme. Dunque, non esiste una separazione netta tra componenti principali informative e non informative e la struttura dei dati non è fortemente concentrata in poche direzioni. Si ha, inoltre, ridondanza tra le variabili. <br> In questi casi, la scelta del numero di componenti principali non è guidata da una discontinuità strutturale, ma da criteri quantitivi sulla varianza cumulativa.  <br>
Osservando la curva della varianza cumulata (grafico sulla destra), si nota che il raggiungimento della soglia del 90% viene richiede 6 componenti principali (91.93% di varianza spiegata cumulata), mentre quella del 95% ne richiede 7 (varianza spiegata cumulata pari al 96.01%).

In [ ]:
#inizializzazione dell'oggetto PCA
pca_percentage = 0.95
pca= PCA(pca_percentage)
#"Fit" dell'oggetto PCA 
X_trainval_pca = pca.fit_transform(X_trainval_scaled_tv)
X_test_pca = pca.transform(X_test_scaled_tv)
print("Numero PC: {}".format(pca.n_components_))
print("% Varianza totale spiegata: {}".format(pca.explained_variance_ratio_.sum()))
print(f"Shape training set dopo PCA: {X_trainval_pca.shape}")
print(f"Shape test set dopo PCA: {X_test_pca.shape}")

La pca con soglia al 95% seleziona in maniera automatica le prime 7 componenti principali che, complessivamente, spiegano il 96.01% della varainza totale. La dimensionalità del dataset è, dunque, ridotta da 10 a 7 feature, con una perdita di informazione pari al 4% circa. Training+validation e test set vengono trasformati in questo nuovo spazio di dimensione ridotta.

Si procede con la visualizzazione della proiezione del training+validation set sulle prime due componenti principali, colorando i punti in base alla classe. <bf>
Il biplot permette di valutare visivamente la separabilità delle classi nello spazio ridotto e l'orientamento delle feature originali rispetto alle componenti

In [ ]:
colors = {0: "steelblue", 1: "tomato"}
labels = {0: "gamma (g)", 1: "hadron (h)"}
 
# Score plot semplice
plt.figure(figsize=(6, 6))
for cls in [0, 1]:
    mask = y_trainval == cls
    plt.scatter(X_trainval_pca[mask, 0], X_trainval_pca[mask, 1],
                c=colors[cls], label=labels[cls], alpha=0.3, s=10)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Score plot PCA – PC1 vs PC2")
plt.legend()
plt.show()
 
# Biplot + loading graph
fig, ax = plt.subplots(1, 2, figsize=(10, 5))
 
loadings      = pca.components_          # shape (n_components, n_features)
feature_names = X.columns.tolist()
 
# — Biplot (ax[0])
for cls in [0, 1]:
    mask = y_trainval == cls
    ax[0].scatter(X_trainval_pca[mask, 0], X_trainval_pca[mask, 1],
                  c=colors[cls], label=labels[cls], alpha=0.2, s=8)
 
scale = 3
for i, feat in enumerate(feature_names):
    ax[0].arrow(0, 0,
                loadings[0, i] * scale, loadings[1, i] * scale,
                color='black', alpha=0.7,
                head_width=0.05, length_includes_head=True)
    ax[0].text(loadings[0, i] * scale * 1.15,
               loadings[1, i] * scale * 1.15,
               feat, fontsize=8, ha='center')
 
ax[0].set_xlabel("PC1")
ax[0].set_ylabel("PC2")
ax[0].set_title("Biplot PCA con loading vectors")
ax[0].legend()
 
# — Loading graph (ax[1])
for i in range(pca.n_features_in_):
    ax[1].plot([0, pca.components_[0, i]], [0, pca.components_[1, i]],
               label=feature_names[i])
 
ax[1].scatter(pca.components_[0, :], pca.components_[1, :], c='k')
ax[1].legend(bbox_to_anchor=(1.05, 1), fontsize='x-small')
ax[1].set_title('Loading Graph – 2 PC')
ax[1].set_xlabel('PC1')
ax[1].set_ylabel('PC2')
ax[1].grid()
 
plt.tight_layout()
plt.show()
 
 

La proiezione sulle prime due componenti principali mostra una sovrapposizione significativa tra le classi gamma (blu) e hadron (rosso): nel piano PC1-PC2, non sono linearmente separabili. La classe gamma tende a concentrarsi leggermente verso valori negativi di PC1, mentre la classe hadron risulta più dispersa. La separazione parziale suggerisce che le prime due componenti siano da sole insufficienti a discriminare in maniera efficiente le classi. <br>
Il biplot con i loading vectors (biplot, grafico in basso a destra) mostra come vengono proiettate le feature originali sul piano PC1-PC2. *fLenght*, *fWidth*, *fSize* puntano nella stessa direzione di crescta di PC1, alla quale forniscono contributi positivi. Ciò è indicativo di una forte correlazione reciproca, dovuta al fatto che misurano grandezze geometriche dell'ellisse legate alla dimensione complessiva dello sciame. *fDist* punta nella stessa direzione di crescita di PC1 ed è quasi parallela ad essa. *fConc* e *fConc1*, invece, puntano in direzione opposta, confermando la correlazione negativa tra la concentrazione dei pixel e la dimensione dell'ellisse. Si nota che *fAsym* e *fM3Long* forniscono un contributo positivo rilevante su PC2, mentre *fAlpha* negativo. Nel caso di *fM3Long*, esso è quasi allinearo a PC2. <br>
Il loading graph (grafico in basso a sinistra) mostra la posizione dei vettori delle feature nel piano PC1-PC2 in maniera più "pulita" rispetto allo scree plot. 

Estensione della visualizzazione alla terza componente principale per catturare ulteriore varianza e visualizzare se la separazione tra classi migliora in uno spazio 3D:

In [ ]:
#grafico PCA 3D
# Creiamo un DataFrame per comodità (Plotly lavora meglio con i DataFrame)
fig_3d = plt.figure(figsize=(8, 7))
ax_3d = fig_3d.add_subplot(111, projection='3d')

for cls in [0, 1]:
    mask = y_trainval == cls
    ax_3d.scatter(
        X_trainval_pca[mask, 0],
        X_trainval_pca[mask, 1],
        X_trainval_pca[mask, 2],
        c=colors[cls],
        label=labels[cls],
        alpha=0.3,
        s=10
    )
ax_3d.set_xlabel("PC1")
ax_3d.set_ylabel("PC2")
ax_3d.set_zlabel("PC3")
ax_3d.set_title("Proiezione PCA 3D")
ax_3d.legend()

#per aggirare i problemi locali di JupyterLab con plotly
pio.renderers.default = "notebook_connected"

# Loading
fig_l = go.Figure()
for i in range(pca.n_features_in_):
    # Aggiungiamo una linea per ogni feature
    fig_l.add_trace(go.Scatter3d(
        x=[0, loadings[0, i]],
        y=[0, loadings[1, i]],
        z=[0, loadings[2, i]],
        mode='lines+markers+text',
        name=feature_names[i],
        text=["", feature_names[i]], # Mostra il nome solo sulla punta
        textposition="top center"
    ))

fig_l.update_layout(
    title="Loading Graph 3D",
    scene=dict(
        xaxis_title='PC1',
        yaxis_title='PC2',
        zaxis_title='PC3'
    ),
    legend=dict(font=dict(size=10))
)

fig_l.show()

Nemmeno la proiezione tridimensionale su PC1, PC2, PC3 rivela una separazione netta tra le classi: le distribuzioni di gamma e hadron rimangono ampiamente sovrapposte lungo tutte e tre le direzioni. Si osserva, inoltre, che gamma è leggermente più densa verso valori negativi di PC3 (si veda la loading fortemente negativa di *fDist* su PC3). <br> 
Questa visualizzazione grafica conferma che il confine di decisione tra le due classi non è lineare e motiva l'utilizzo di un kernel RBF (Radial Basis Function) per l'SVM.

In [ ]:
loadings_df = pd.DataFrame(pca.components_, columns = feature_names, index = [f"PC{i+1}" for i in range(pca.n_components_)])
plt.figure(figsize=(10,7))
plt.imshow(loadings_df.values, cmap='coolwarm', aspect='auto', vmin=-1, vmax=1)
plt.colorbar(label='Loading')
plt.xticks(range(len(feature_names)), feature_names, rotation=45, ha='right')
plt.yticks(range(pca.n_components_), loadings_df.index)
plt.title("Heatmap dei Loadings PCA")
plt.tight_layout()
plt.show()
print(loadings_df.round(3))                                                                              

La heatmap dei loadings mostra il contributo di ciascuna feature originale a ciacuna componente principale. Valori elevati in modulo indicano che quella feature contribuisce maggiormente alla costruzione della componente. 
- PC1 è dominata da *fSize*, *fLenght*, *fWidth* con segno positivo e da *fConc* e *fConc1* con segno negativo; come affermato in precedenza, ciò è dovuto alla correlazione negativa tra la dimensione dello sciame di particelle e la concentrazione dei pixel;
- PC2 è dominata positivamente da *fM3Long* e *fAsym* e negativamente da *fAlpha*; ciò è dovuto all'asimmetria longitudinale dello sciame;
- PC3 è dominata da *fAlpha* con segno positivo e da *fDist* con segno negativo; lega, dunque, la distanza dall'origine all'angolo dell'asse maggiore, suggerndo una relazione inversa tra le due feature;
- PC4 è dominata positivamente da *fM3Trans*, il terzo momento trasversale è rappresentato in maniera quasi perfetta;
Le ultime tre componenti principali catturano la varianza residua mista, con contributi più distributi tra le feature.

Tramite dei barplot si visualizza il contributo delle features originali per le prime tre PC.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(18, 5), sharey=True) 
titles = ['PC1', 'PC2', 'PC3']
x_pos = np.arange(pca.n_features_in_)
for i in range(3):
    ax[i].bar(x_pos, pca.components_[i, :])
    ax[i].set_xticks(x_pos, labels = features, rotation=45)
    ax[i].set_title(titles[i], fontweight='bold')
    ax[i].grid(axis='y', linestyle='--', alpha=0.7)
    ax[i].axhline(0, color='black', linewidth=0.8)
fig.suptitle('PCA Loadings Comparison', fontsize=14, fontweight='bold', y=1.05)
plt.tight_layout()
plt.show()

I risultati osservati nell'heatmap trovano conferma negli istogrammi, in cui la lettura dei contributi relativi è molto più immediata.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
splits = [
    (X_trainval_pca, y_trainval, "Training+validation set"),
    (X_test_pca,  y_test,  "Test set")
]
for ax, (X_proj, y_split, title) in zip(axes, splits):
    for cls in [0, 1]:
        mask = y_split == cls
        ax.hist(
            X_proj[mask, 0],
            bins=60, alpha=0.5,
            color=colors[cls],
            label=labels[cls],
            density=True
        )
    ax.set_title(title)
    ax.set_xlabel("PC1")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Densità")
plt.suptitle("PCA — PC1: separazione delle classi su training, validation e test set", fontsize=13)
plt.tight_layout()
plt.show()

Questi grafici mostrano la distribuzione delle classi su PC1 nel training+validation e nel test set. Si osserva che le distribuzioni sono stabili nei due set: la forma, disposizione dei picchi e la sovrapposizione tra classi rimangono coerenti, confermando la buona rappresentatività della divisione. <br> Si osserva, inoltre, che la sovrapposizione lungo la prima componente principale è molto ampia, confermando che PC1 singolarmente non è sufficiente a classificare in maniera efficiente eventi gamma e adronici. 

# Riduzione della dimensionalità: FDA e MDA

## Fisher Discriminant Analysis (FDA)

La *Fisher Discriminant Analysis (FDA)* è una tecnica supervisionata di
riduzione della dimensionalità utile per la classificazione, dal momento che il suo obiettivo
consiste nel determinare una proiezione lineare dei dati capace di
massimizzare la separazione tra classi. A differenza della PCA, la FDA
sfrutta l'informazione supervisionata contenuta nelle \"etichette\" dei
dati per individuare una direzione ottimale di discriminazione.\
Sia
$S = \{(x_i, y_i)\} \subset \mathbb{R}^n \times \{\mathcal{C}_1, \mathcal{C}_2\}$
un insieme di dati etichettati appartenenti a due classi
($\mathcal{C}_1 \quad  \text{e} \quad \mathcal{C}_2$). L'obiettivo della
FDA è trovare un versore $v \in \mathbb{R}^n$, tale che la proiezione
$\pi(x) = v^\top x$ produca una rappresentazione in cui le due classi
risultino il più possibile separate.\
Per raggiungere tale scopo si introducono le medie campionarie delle due
classi:
$$
\mu_1 = \frac{1}{N_1} \sum_{x_i \in \mathcal{C}_1} x_i, \qquad \mu_2 = \frac{1}{N_2}\sum_{x_i \in \mathcal{C}_2}x_i,
$$
dove $N_1$ e $N_2$ indicano rispettivamente il numero di esempi
appartenenti alla prima e alla seconda classe.
Proiettando le medie si ottiene:
$$
\tilde \mu_1 = v^\top \mu_1, \qquad \tilde \mu_2 = v^\top \mu_2.
$$ 
Dal
momento che occorre anche tenere in considerazione l'informazione sulla
varianza (dispersione) all'interno di ciascuna classe, si definiscono le
*matrici di scatter (scatter matrix)*:
$$S_k = \sum_{x_i \in \mathcal{C}_k} (x_i - \mu_k)(x_i - \mu_k)^\top, \quad k = 1,2.$$
Si definisce, quindi, la *matrice di scatter totale intra-classe (within
the classes scatter matrix)*: 
$$
S_w = S_1 + S_2.
$$ 
Infine, proiettando
queste matrici lungo la direzione individuata da $v$ si ottiene:
$$
\tilde S_i = v^\top S_i v, \quad i = 1,2,
$$
$$
\tilde S_w = \tilde S_1 + \tilde S_2 = v^\top S_w v.
$$ 
Il criterio di
Fisher si pone l'obiettivo di massimizzare e il rapporto tra la
separazione inter-classe e la dispersione intra-classe nello spazio di arrivo della proiezione. La funzione
obiettivo è, dunque:
$$
J(v) = \frac{(\tilde \mu_1 - \tilde \mu_2)^2}{\tilde S_1 + \tilde S_2} = \frac{(v^\top( \mu_1 - \mu_2))^2}{\tilde S_1 + \tilde S_2}  = \frac{v^\top S_b v}{v^\top S_w v},
$$
dove $S_b = (\mu_1 - \mu_2)(\mu_1 - \mu_2)^\top$ è la *matrice di
scatter inter-classe (between the classes scatter matrix)*.\
Per determinare la direzione ottimale $v$ si risolve il problema:
$$
\max_{v \neq 0} \; J(v),
$$
ricordando che, poiché il funzionale $(J)$ è invariante rispetto alla scala di $(v)$, è possibile determinare una soluzione generale e successivamente normalizzarla per ottenere un versore.\
Derivando $J$ rispetto a $v$ e imponendo
l'annullamento del gradiente si ottiene:
$$
S_b v - \frac{v^\top S_b v}{v^\top S_w v} S_w v = 0.
$$ Ponendo
$$
\lambda := \frac{v^\top S_b v}{v^\top S_w v} \in \mathbb{R},
$$ segue
la relazione: 
$$
S_b v = \lambda S_w v,
$$ 
che rappresenta un problema
agli autovalori generalizzato. Assumendo che $S_w$ sia invertibile, si
può riscrivere il problema nella forma standard:
$$
S_w^{-1} S_b v = \lambda v,
$$ 
ovvero un problema agli autovalori della
matrice $S_w^{-1} S_b$. La soluzione ottima è data dall'autovettore
associato al massimo autovalore (eventualmente normalizzato). Osservando che
$\forall x \in \mathbb{R}^n, \quad (\mu_1 - \mu_2)^\top x = \alpha \in \mathbb{R}$, si ha che
$S_b x = (\mu_1 - \mu_2)(\mu_1 - \mu_2)^\top x = \alpha (\mu_1-\mu_2)$,
ovvero che $S_b x$ punta sempre nella stessa direzione di
$\mu_1 - \mu_2$. Di conseguenza, si ottiene che la direzione
discriminante $v^*$ è tale per cui:
$$
v^\ast \propto S_w^{-1}(\mu_1 - \mu_2).
$$ 
In conclusione, la Fisher
Discriminant Analysis si configura come una tecnica supervisionata di
riduzione della dimensionalità che integra in modo naturale la struttura
geometrica e l'informazione statistica delle classi. La sua
formulazione, basata su un problema agli autovalori generalizzato,
evidenzia come la direzione ottimale di proiezione emerga da un
bilanciamento tra la separazione tra le medie e la \"compattezza\"
intra-classe.

## Multiple Discriminant Analysis (MDA)

La *Multiple Discriminant Analysis (MDA)* è una generalizzazione della
Fisher Discriminant Analysis al caso multiclasse. L'obiettivo che si pone consiste
nel determinare una trasformazione lineare $z = V^\top x$, dove
$V \in \mathbb{R}^{n \times d}, \quad d < K$ (con $K$ il numero di
classi) e tale che le osservazioni proiettate risultino il più possibile
separabili nello spazio ridotto.\
Analogamente al caso binario, si introducono la within the classes
scatter matrix (di rango al più $(K-1)$)
$$
S_w = \sum_{i = 1}^K S_i = \sum_{i = 1}^K \sum_{x_j \in \mathcal{C}_i} (x_j - \mu_i)(x_j - \mu_i)^\top\ \in \mathbb{R}^{n \times n}
$$
e la between the classes scatter matrix
$$
S_b = \sum_{i = 1}^K n_i (\mu_i - \mu)(\mu_i - \mu)^\top \in \mathbb{R}^{n \times n},
$$
dove $\mu_i$ rappresenta la media della classe $i$, $ n_i$ il numero di
elementi del dataset appartenenti alla classe $i$ e $\mu$ la media
globale del dataset.\
Anche nell'MDA si massimizza il rapporto tra la variabilità interclasse
e la dispersione intraclasse, espresso da:
$$
J(V) = \frac{\det(V^\top S_b V)}{\det(V^ \top S_w V)}.
$$ 
Anche in
questo caso, si ottiene un problema agli autovalori generalizzato. Nel
caso in cui la matrice $S_w$ sia invertibile, il problema si riduce alla
ricerca degli autovettori e degli autovalori della matrice
$S = S_w^{-1}S_b$. Si identificano, infine, i $d$ autovettori
$v_1, \dots, v_d$ associati agli autovalori maggiori. Questi vettori
costituiscono le colonne della matrice $V$, definendo il sottospazio
discriminante ottimale in cui la separazione tra classi risulta
massimizzata.

## FDA e MAGIC Gamma Telescope
Dal momento che il problema in esame è di classificazione binaria, si applica la Fisher Discriminant Analysis (FDA) e non la Multiple Fisher Discriminant Analysis (MDA) generica.<br> Per fare ciò, viene utilizzato il codice fornito dal Professore Francesco Della Santa (FisherDA.py)# Riduzione della dimensionalità: FDA e MDA



In [ ]:
fda = MDA()
fda.fit(X_trainval_scaled_tv, y_trainval.values)
X_trainval_fda = fda.transform(X_trainval_scaled_tv)
X_test_fda = fda.transform(X_test_scaled_tv)
print(f"Dimensioni training+validation set: {X_trainval_fda.shape}")
print(f"Dimensioni test set: {X_test_fda.shape}")
print(f"Autovalore della componente discriminante: {fda.eigenvalues_[0].real:.4f}")

L'autovalore della componente principale (0.4791) rappresenta il rapporto tra la varianza interclasse (between-the-classes) e intraclasse (within-the-classes) lungo la direzione ottimale trovata dalla FDA. Tale valore positivo conferma che la dirizione discriminante separa le due classi in maniera significativa.

La FDA produce una sola componente discriminante e, di conseguenza, la visualizzazione più informativa è un istograma sovrapposto delle due classi proiettate su un unico asse discriminante. <br> Una separazione netta tra i picchi indica un buon potere discriminante.

È possibile confrontare la separazione ottenuta dalla FDA con quella della prima componente principale della PCA. Ci si aspetta una separazione maggiore nel caso della Fisher Discriminant Analysis dal momento che si tratta di un modello supervisionato.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for cls in [0, 1]:
    mask = y_trainval == cls
    axes[0].hist(
        X_trainval_fda[mask, 0].real,
        bins=60, alpha=0.5,
        color=colors[cls],
        label=labels[cls],
        density=True
    )
axes[0].set_xlabel("Componente discriminante (FDA)")
axes[0].set_ylabel("Densità")
axes[0].set_title("FDA — Componente 1")
axes[0].legend()

for cls in [0, 1]:
    mask = y_trainval == cls
    axes[1].hist(
        X_trainval_pca[mask, 0],
        bins=60, alpha=0.5,
        color=colors[cls],
        label=labels[cls],
        density=True
    )
axes[1].set_xlabel("PC1")
axes[1].set_ylabel("Densità")
axes[1].set_title("PCA — PC1")
axes[1].legend()

plt.suptitle("Confronto FDA vs PCA: separazione delle classi", fontsize=13)
plt.tight_layout()
plt.show()

Nella FDA le distribuzioni di gamma (blu) e hadron (rosso) risultano parzialmente sovrapposte, ma con una separazione visibile. La classe g è concentrata verso valori negativi, mentre la classe h si estende maggiormente lungo valori positivi. <br> La sovrapposizione indica che, nonostante l'applicazione della FDA, il decision boundary non è netto e che il problema non è linearmente separabile nemmeno nello spazio della FDA.

Viene confermato quanto affermato in precedenza: la FDA è migliore per classificazione. Infatti, nel grafico della FDA (sulla sinistra), le due distribuzioni mostrano picchi più distanziati e una sovrapposizione meno estesa rispetto alla PC1 (figura sulla destra), dove si ha una separazione lievissima. <br>
Questo risultato non è inaspettato, ma conseguenza diretta della costruzione dei due metodi: la FDA ottimizza la separazione tra classi, mentre la PCA la varianza globale (senza prendere in considerazione le "etichette").

È posibile tentare di valutare quantitivamente la separazione tra classi. Dal momento che l'obiettivo della separazione è duplice e comprende:
- la massimizzazione della distanza tra la media tra le due classi
- la minima dispersione (varianza) possibile all'interno delle due classi
  
Quindi, possiamo definire un coefficiente *f_sep* direttamente proporzionale a $(\mu_g - \mu_h)^2$, inversamente propozionale a $(\sigma_g^2 + \sigma_h^2)$ e capace di stimare la separazione tra le classi. A valori alti di *f_sep* corrisponde elevata separabilità.

In [ ]:
mu_g_fda =X_trainval_fda[y_trainval.values == 0].mean()
var_g_fda = X_trainval_fda[y_trainval.values==0].var()
mu_h_fda =X_trainval_fda[y_trainval.values ==1].mean()
var_h_fda =X_trainval_fda[y_trainval.values ==1].var()
mu_g_pca =X_trainval_pca[y_trainval.values == 0,0].mean() #uso solo la prima componente
var_g_pca = X_trainval_pca[y_trainval.values==0,0].var()
mu_h_pca =X_trainval_pca[y_trainval.values ==1,0].mean()
var_h_pca =X_trainval_pca[y_trainval.values ==1,0].var()
f_sep_fda = (mu_g_fda - mu_h_fda)**2/(var_h_fda + var_g_fda)
f_sep_pca = (mu_g_pca - mu_h_pca)**2/(var_h_pca + var_g_pca)
print(f"f_sep - FDA: {f_sep_fda.real:.4f}") #il valore fisicamente significativo è quello reale
print(f"f_sep - PCA componente principale n.1: {f_sep_pca.real:.4f} - pca")

L'indice *f_sep* quantifica in maniera netta la differenza tra i due approcci: a differenza della PCA, la FDA è stata progettata esattamente per massimizzare questa quantità. Il fatto che *f_sep* per la PCA sia molgo più basso significa che le due classi (gamma e hadron) hanno distribuzioni molto simili lungo PC1 e, di conseguenza, si sovrappongono completamente.

Si verifica, infine, che la separazione osservata sul training+validation set si mantenga anche sul test set.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=True)
splits = [
    (X_trainval_fda, y_trainval, "Training+validation set"),
    (X_test_fda,  y_test,  "Test set")
]
for ax, (X_proj, y_split, title) in zip(axes, splits):
    for cls in [0, 1]:
        mask = y_split == cls
        ax.hist(
            X_proj[mask, 0].real,
            bins=60, alpha=0.5,
            color=colors[cls],
            label=labels[cls],
            density=True
        )
    ax.set_title(title)
    ax.set_xlabel("Componente discriminante (FDA)")
    ax.legend(fontsize=8)
axes[0].set_ylabel("Densità")
plt.suptitle("FDA — Separazione delle classi su train+val / test", fontsize=13)
plt.tight_layout()
plt.show()

La separazione tra le classi si mantiene, dunque, stabile e coerente su training+validation e test set: la forma delle distribuzioni, la posizione dei picchi e il grado di sovrapposizione sono analoghi nei due set. Ciò conferma ancora una volta le buone capacità di generalizzazione della componente discriminante di Fisher appresa sul training set.

# Classificazione: LDA e QDA 

La *Linear Discriminant Analysis (LDA)* e la *Quadratic Discriminant
Analysis (QDA)* sono tecniche di classificazione supervisionata
ampiamente utilizzate nel machine learning. Entrambe si basano su un
approccio probabilistico e sul teorema di Bayes, mirando ad assegnare
ogni nuova osservazione alla proprietà a posteriori maggiore. Detto
$\mathcal{C} = \{1, \dots, K\}$ l'insieme delle classi possibili e
$x \in \mathbb{R}^n$, l'obiettivo è stimare la probabilità a posteriori
$P(Y=k| X = x)$, dove $Y$ rappresenta la variabile casuale associata
alla classe $k \in \mathcal{C}$.\
Dal *teorema di Bayes* segue:
$$P(Y = k | X = x) = \frac{P(X = x | Y = k) \cdot P(Y=k)}{P(X = x)} =\frac{\pi_k f_k(x)}{\sum_{l = 1}^K \pi_l f_l(x)},$$
con

- $\pi_k = P(Y=k) \in (0,1)$, probabilità a priori della classe $k$;

- $f_k(x)$ è la densità condizionata delle feature nella classe $k$

La classificazione viene effettuata assegnando $x$ alla classe che
massimizza $\pi_kf_k(x)$.\
Sia la LDA che la QDA si basano sull'assunzione che i dati delle classi
abbiano distribuzione normale
($X \mid Y = k \sim \mathcal{N}(\mu_k, \Sigma)$), la differenza tra i
due algoritmi interessa l'ipotesi sulla matrice di covarianza tra
classi.

## Linear Discriminant Analysis (LDA)

L'ipotesi alla base della LDA è l'omoschedasticità delle classi, ovvero
il fatto che le classi abbiano la stessa matrice di varianza e covarianza
$\Sigma$. Detto $\mu_k$ il vettore della medie della classe k, la
funzione di densità gaussiana multivariata è:
$$f_k(x)=\frac{1}{(2\pi)^{n/2}\sqrt{|\Sigma|}}e^{-\frac{1}{2}(x - \mu_k)^T \Sigma^{-1}(x-\mu_k)}.$$
Sostituendo questa espressione nella formula di Bayes e trascurando i
termini comuni a tutte le classi, si ottiene una funzione discriminante:
$$
\delta_k(x) = x^\top \Sigma^{-1}\mu_k - \frac{1}{2} \mu_k^\top \Sigma^{-1} \mu_k + \log \pi_k.
$$
L'osservazione $x$ viene assegnata alla classe:
$$
\arg \max_{k \in \mathcal{C}} \delta_k(x).
$$ 
I decision boundary così
ottenuti sono iperpiani lineari, da cui il nome *Linear Discriminant
Analysis*.\
Tra i vantaggi di questo metodo rientrano il fatto che
richieda un numero relativamente ridotto di parametri, che funzioni
anche con dataset piccoli, l'efficienza computazionale e la riduzione
del rischio di overfitting. Tuttavia, l'ipotesi di omoschedasticità può
risultare troppo restrittiva nelle applicazioni.

### FDA e LDA

La *Fisher Discriminant Analysis (FDA)* è un metodo geometrico di proiezione lineare, basato sulla ricerca di una direzione che massimizzi la separazione tra le classi e minimizzi la dispersione all'interno di ciascuna di esse. In tale formulazione non viene assunta alcuna distribuzione probabilistica per i dati: l'obiettivo consiste esclusivamente nell'ottimizzazione del criterio di Fisher
$$
J(v)=\frac{v^\top S_b v}{v^\top S_w v},
$$
dove ($S_b$) e ($S_w$) rappresentano rispettivamente le matrici di scatter between e within the classes.
Tuttavia, la FDA ammette un'interpretazione probabilistica molto significativa. In particolare, la *Linear Discriminant Analysis (LDA)* assume che le osservazioni appartenenti alla classe ($k$) siano generate da una distribuzione gaussiana multivariata
$$
X \mid Y=k \sim \mathcal N(\mu_k,\Sigma),
$$
dove ($\mu_k$) è la media della classe e ($\Sigma$) è la matrice di covarianza comune a tutte le classi.
Sotto tali ipotesi, la classificazione ottimale in senso bayesiano si ottiene confrontando le probabilità a posteriori delle diverse classi. Grazie all'assunzione di covarianza comune, le frontiere decisionali risultano lineari e la direzione discriminante assume la forma
$$
w = \Sigma^{-1}(\mu_1-\mu_2).
$$
D'altra parte, la soluzione del problema di ottimizzazione della FDA nel caso binario è proporzionale a
$$
v^\star = S_w^{-1}(\mu_1-\mu_2).
$$
Poiché la matrice di scatter intra-classe ($S_w$) può essere interpretata come una stima campionaria della covarianza comune ($\Sigma$), le due espressioni coincidono a meno di un fattore di scala.
Di conseguenza, FDA e LDA possono essere interpretate come due formulazioni equivalenti dello stesso principio discriminante: la prima nasce da un criterio puramente geometrico di massimizzazione della separazione tra classi, mentre la seconda deriva da un modello probabilistico generativo basato su distribuzioni gaussiane multivariate.

## Quadratic Discriminant Analysis (QDA)

Rilassando l'ipotesi di omoschedasticità, si ottiene la *quadratic
discriminant analysis* che assume:
$$
X | Y = k \sim \mathcal{N}(\mu_k, \Sigma_k),
$$ 
con $\Sigma_k$ matrice
di varianza e covarianza della classe $k$.\
La funzione discriminante diventa:
$$
\delta_{k}(x) = -\frac{1}{2}(x-\mu_k)^T \Sigma^{-1}_k (x-\mu_k) + \log \pi_k - \frac{1}{2}\log|\Sigma_k|\,, \quad \forall \ k=1,\ldots , c\,.
$$

La presenza di termini quadratici in $x$ fa sì che le frontiere
decisionali siano quadratiche e non lineari come nella LDA. Inoltre, ciò
fa sì che la QDA risulti più flessibile e questo pregio risalta
soprattutto nella modellazione di strutture più complesse nei dati.\
Tuttavia, la QDA richiede dataset più grandi, è più soggetta ad
overfitting e rischia di diventare instabile quando il numero di feature
è elevato.

## LDA, QDA e MAGIC Gamma Telescope
Nell'EDA gli istogrammi e i boxplot mostrano distribuzioni asimmetriche e bimodali per molte feature, suggerendo la non normalità delle classi.<br> Tuttavia, per scopi illustrativi, si suppone che all'interno delle classi i campioni si distribuiscano secondo una distribuzione normale, per tentare di applicare la QDA e l'LDA. <br>Per prima cosa, si valuta empiricamente l'assunzione di omoschedasticità tramite la norma di Frobenius della differenza tra le matrici di covarianza delle due classi.

In [ ]:
#in questo caso si utilizzerà il validation set per scegliere quale sia meglio per le visualizzazioni
cov_g = np.cov(X_scaled[y.values==0].T) 
cov_h = np.cov(X_scaled[y.values==1].T)
diff_fro = np.linalg.norm(cov_g - cov_h, 'fro')
print(f"Differenza tra le matrici di covarianza delle due classi in norma di Frobenius: {diff_fro:4f}")
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
im0 = axes[0].imshow(cov_g, cmap='coolwarm')
axes[0].set_title("Matrice di covarianza — gamma (g)")
plt.colorbar(im0, ax=axes[0])
im1 = axes[1].imshow(cov_h, cmap='coolwarm')
axes[1].set_title("Matrice di covarianza — hadron (h)")
plt.colorbar(im1, ax=axes[1])
im2 = axes[2].imshow(np.abs(cov_g - cov_h), cmap='Reds')
axes[2].set_title(f"Differenza |Σ_g - Σ_h|  (Frobenius: {diff_fro:.2f})")
plt.colorbar(im2, ax=axes[2])
plt.tight_layout()
plt.show()

Il valore $||\Sigma_g - \Sigma_h||_F = 5.6$ indica che le matrici di covarianza tra le due classi sono diverse tra loro, quindi l'omoschedasticità non è soddisfatta. <bf>
Ciò sembrerebbe suggerire l'applicazione di una classificazione di tipo QDA, servendosi di una matrice di covarianza separata per oguna delle classi. Tuttavia, dalla documentazione ufficiale del dataset, occorre utilizzare la metrica AUC sul validation set, che in alcuni casi è più robusta agli outlier. <br> Occorre, dunque, tentare entrambe le vie per comprendere quale delle due sia effettivamente la migliore.

In [ ]:
lda = LDA() #inizializzazione dell'oggetto LDA
lda.fit(X_train_scaled, y_train)
qda = QDA() #inizializzazione dell'oggetto QDA
qda.fit(X_train_scaled,y_train)
#valutazione in metrica AUC sul validation set
auc_lda_val = roc_auc_score(y_val, lda.predict_proba(X_val_scaled)[:, 1])
auc_qda_val = roc_auc_score(y_val, qda.predict_proba(X_val_scaled)[:, 1])
print(f"AUC LDA val: {auc_lda_val:.4f}")
print(f"AUC QDA val: {auc_qda_val:.4f}")

I risultati sul validation set confermano che la QDA supera la LDA, coerentemente con la norma di Frobenius calcolata in precedenza sull'intero dataset. Il decision boundary quadratico della QDA, dunque, si adatta meglio alla reale struttura dei dati. <br> Dal momento che entrambi i modelli presentano un'elevata AUC, anche la LDA ha una buona capacità discriminante.

In [ ]:
auc_qda_t = roc_auc_score(y_test, qda.predict_proba(X_test_scaled)[:, 1])
print(f"AUC QDA test set : {auc_qda_t:.4f}")

Si procede con alcune visualizzazioni del modello QDA.

In [ ]:
#curva roc, come da documentazione, applicaata al test set
# ROC con gamma (segnale) come classe positiva:  proba[:,0], pos_label=0.
# L'AUC e' invariata per simmetria.
fpr_qda_t, tpr_qda_t, _ = roc_curve(y_test, qda.predict_proba(X_test_scaled)[:, 0], pos_label=0)
fpr_lda_t, tpr_lda_t, _ = roc_curve(y_test, lda.predict_proba(X_test_scaled)[:, 0], pos_label=0)
auc_lda_t = roc_auc_score(y_test, lda.predict_proba(X_test_scaled)[:,1])
plt.figure(figsize=(7, 6))
plt.plot(fpr_qda_t, tpr_qda_t, label=f"QDA  (AUC = {auc_qda_t:.4f})", color='tomato')
plt.plot(fpr_lda_t, tpr_lda_t, label=f"LDA  (AUC = {auc_lda_t:.4f})", color='steelblue', linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — LDA vs QDA (test set)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
class_names = ['gamma (g)', 'hadron (h)']
y_pred_qda_test = qda.predict(X_test_scaled)
y_proba_qda_test = qda.predict_proba(X_test_scaled)[:, 1]
y_pred_qda_trainval = qda.predict(X_trainval_scaled)
y_proba_qda_trainval = qda.predict_proba(X_trainval_scaled)[:,1]
df_perf_qda = pd.DataFrame({
    'Accuracy':  [accuracy_score(y_trainval, y_pred_qda_trainval),
                  accuracy_score(y_test,     y_pred_qda_test)],
    'Precision': [precision_score(y_trainval, y_pred_qda_trainval, average='weighted'),
                  precision_score(y_test,     y_pred_qda_test,     average='weighted')],
    'Recall':    [recall_score(y_trainval, y_pred_qda_trainval, average='weighted'),
                  recall_score(y_test,     y_pred_qda_test,     average='weighted')],
    'AUC':       [roc_auc_score(y_trainval, y_proba_qda_trainval),
                  roc_auc_score(y_test,     y_proba_qda_test)],
}, index=['training+validation', 'test'])
display(df_perf_qda.round(4))
#confusion matrix
cmat_qda           = confusion_matrix(y_test, y_pred_qda_test)
cmat_qda_norm_true = confusion_matrix(y_test, y_pred_qda_test, normalize='true')
cmat_qda_norm_pred = confusion_matrix(y_test, y_pred_qda_test, normalize='pred')
#plotting
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(cmat_qda,
    display_labels=class_names).plot(ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title("Valori assoluti")
ConfusionMatrixDisplay(cmat_qda_norm_true,
    display_labels=class_names).plot(ax=axes[1], colorbar=False, cmap='Blues', values_format='.3f')
axes[1].set_title("Normalizzata su vere classi (recall)")
ConfusionMatrixDisplay(cmat_qda_norm_pred,
    display_labels=class_names).plot(ax=axes[2], colorbar=False, cmap='Blues', values_format='.3f')
axes[2].set_title("Normalizzata su classi predette (precision)")
plt.suptitle("Matrici di confusione — QDA (test set)", fontsize=13)
plt.tight_layout()
plt.show()

I valori delle metriche valutati sul training+validation e sul test set, invece, sono coerenti; si ha, quindi, assenza di overfitting. <br>Le matrici di confusione della QDA sul test set mostrano un comportamento asimmetrico tra le due classi. Lo squilibrio riflette lo sbilanciamento del dataset in quanto il modello tende a classificare i campioni come appartententi alla classe maggioritaria (g), penalizzando il riconoscimento degli eventi adronici. <br>
La matrice di Recall (normalizzata su vere classi) rappresenta la capacità del modello di identificare correttamente i membri appartenenti a una specifica classe reale (*True Positive Rate*):
* gamma: ottima sensibilità. Solo il 5.7% dei fotoni gamma viene erroneamente classificato come adrone.
* hadron: performance critica. Il modello fallisce nell'identificare il 50.7% degli adroni reali, classificandoli erroneamente come gamma (falsi negativi per la classe adronica). Di fatto, la classificazione degli adroni è paragonabile a un lancio di moneta casuale.
La matrice di Precision (normalizzata su classi predette) rappresenta l'affidabilità delle predizioni effettuate dal modello:
* Quando il modello predice gamma, la previsione è corretta nel 77.4%dei casi (influenzata dall'alto numero di adroni scambiati per gamma, $1188$).
* Quando il modello predice hadron, la previsione è corretta nell'82.3% dei casi (poiché i falsi positivi gamma inseriti in questa classe sono pochissimi, solo $248$).

### LDA e FDA
Nonostante su questo dataset, la QDA performi meglio dell' LDA, è interessante confrontare i risultati ottenuti da quest'ultimo modello con quelli dell'FDA.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 5))
#FDA: istogramma 1D con soglia 
# la soglia LDA nello spazio FDA è il punto dove le due gaussiane si intersecano
# corrisponde alla media delle medie delle due classi proiettate
mu0_fda = X_trainval_fda[y_trainval == 0, 0].real.mean()
mu1_fda = X_trainval_fda[y_trainval == 1, 0].real.mean()
threshold_fda = (mu0_fda + mu1_fda) / 2

for cls in [0, 1]:
    mask = y_trainval == cls
    axes[0].hist(
        X_trainval_fda[mask, 0].real,
        bins=60, alpha=0.5,
        color=colors[cls], label=labels[cls], density=True
    )
axes[0].axvline(threshold_fda, color='black', linestyle='--',
                linewidth=1.5, label=f'Soglia = {threshold_fda:.3f}')
axes[0].set_xlabel("Componente discriminante (FDA)")
axes[0].set_ylabel("Densità")
axes[0].set_title("FDA — Proiezione 1D\n(soglia di decisione)")
axes[0].legend(fontsize=8)
axes[0].grid(alpha=0.3)

#2. Confronto loadings FDA vs coefficienti LDA 
fda_coef = pd.Series(fda.eigenvectors_[:, 0].real, index=X.columns)
lda_coef = pd.Series(lda.coef_[0], index=X.columns)

# normalizziamo per confronto (stessa scala)
fda_norm = fda_coef / np.abs(fda_coef).max()
lda_norm = lda_coef / np.abs(lda_coef).max()

x_pos = np.arange(len(X.columns))
width = 0.35

axes[1].bar(x_pos - width/2, fda_norm, width,
            label='FDA', color='steelblue', edgecolor='black', alpha=0.8)
axes[1].bar(x_pos + width/2, lda_norm, width,
            label='LDA', color='tomato', edgecolor='black', alpha=0.8)
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].set_xticks(x_pos)
axes[1].set_xticklabels(X.columns, rotation=45, ha='right', fontsize=8)
axes[1].set_ylabel("Coefficiente normalizzato")
axes[1].set_title("Confronto coefficienti\nFDA vs LDA (normalizzati)")
axes[1].legend()
axes[1].grid(alpha=0.3, axis='y')

# correlazione tra i coefficienti
corr = np.corrcoef(fda_norm, lda_norm)[0, 1]
print(f"Correlazione tra coefficienti FDA e LDA: {corr:.4f}")

Il confronto tra LDA e FDA mostra lo stretto legame tra i due modelli:
- il barplot dei coefficienti mostra che i coefficienti normalizzati di LDA e FDA sono quasi identitici per quasi tutte le feature; le feature più gandi in valore assoluto sono quelle più discriminanti; le discrepanze tra i coefficienti sono dovuti alle formulazioni diverse: l'FDA massimizza il criterio di Fisher (la funzione $J(v)$), mentre la LDA stima le distribuzioni gaussiane delle classi e deriva da esse le direzioni discriminanti; tuttavia, i due metodi convergono alla stessa direzione ottimale;
- la proiezione 1D mostra che la soglia di decisione dell'LDA (la linea tratteggiata) cade tra le medie delle due distribuzioni proiettate, coerentemente con l'assunzione di omoschedasticità dell'LDA. 

# Classificazione: Support Vector Machine (SVM)

Con il termine *Support Vector Machine(s) (SVM)* si indicano algoritmi di
classificazione binaria supervisionata che cercano di separare le
osservazioni appartenenti a classi diverse mediante un iperpiano $\pi$.
Tra tutti gli iperpiani che separano le classi si sceglie quello che
massimizza il *margine*, ovvero la distanza minima tra l'iperpiano e i
punti del training set.

## SVM lineare - Caso linearmente separabile

Consideriamo un problema di classificazione binaria su un dataset $S$
costituito da $m$ esempi indipendenti e identicamente distribuiti
secondo una distribuzione ignota $\mathcal{D}$. Il dataset è della forma
$$
S = \{(x_i, y_i)\}_{i=1}^m,
$$ 
dove $x_i \in \mathbb{R}^n$ e
$y_i \in \{-1, +1\}$. Si assume che $S$ sia *linearmente separabile*,
ovvero che esistano un vettore $w \in \mathbb{R}^n$ (ortogonale
all'iperpiano) e un termine di bias $b \in \mathbb{R}$ tali che
$$
y_i (w^\top x_i + b) > 0, \quad \forall i = 1, \dots, m
$$ 
(ovvero il
segno dell'\"etichetta\" $y_i$ è concorde con quello dell'iperpiano di
parametri $w$ e $b$ valutato in $x_i$) Il margine geometrico associato a
un classificatore lineare è definito come la distanza minima dei punti
dall'iperpiano:
$$
\rho = \min_{i=1,\dots,m} \frac{|w^\top x_i + b|}{\|w\|}.
$$ 
Il
problema di massimizzazione del margine si scrive quindi come:
$$
\rho^\ast =
\max_{w,b}
\left\{
\min_{i=1,\dots,m} \frac{|w^\top x_i + b|}{\|w\|}
\ \text{s.t. } y_i(w^\top x_i + b) > 0
\right\}.
$$ 
Sfruttando l'invarianza di scala della funzione da
minimizzare, 
$$
\frac{|(\alpha w)^\top x_i + \alpha b|}{\|\alpha w\|} =
\frac{|w^\top x_i + b|}{\|w\|}, \quad \forall \alpha \neq 0,
$$ 
è
possibile fissare una normalizzazione tale che:
$$
\min_{i=1,\dots,m} |w^\top x_i + b| = 1.
$$ 
In tal caso il problema di
massimizzazione del margine equivale a:
$$
\max_{w,b} \frac{1}{\|w\|} \quad \text{s.t. } y_i(w^\top x_i + b) \geq 1.
$$
Poiché la funzione obiettivo è monotona, si ottiene il problema
equivalente di minimizzazione: 
$$
\begin{equation*}
\begin{aligned}
& \min_{w,b} && \frac{1}{2}\|w\|^2 \\
& \text{s.t.} && y_i(w^\top x_i + b) \geq 1, \quad \forall i=1,\dots,m.
\end{aligned}
\end{equation*}
$$ 
Trattandosi di un problema di ottimizzazione convessa,
esso ammette un'unica soluzione globale. Introducendo i moltiplicatori
di Lagrange $\alpha_i \geq 0$, si ottiene che il vettore dei pesi può
essere scritto come: 
$$
w = \sum_{i=1}^m \alpha_i y_i x_i,
$$ 
e vale
inoltre il vincolo: 
$$
\sum_{i=1}^m \alpha_i y_i = 0.
$$ 
Le condizioni di
complementarità (KKT) sono:
$$
\alpha_i \big(y_i(w^\top x_i + b) - 1\big) = 0, \quad \forall i=1,\dots,m.
$$
Da cui segue che:
$$
\alpha_i > 0 \;\Rightarrow\; y_i(w^\top x_i + b) = 1.
$$ 
I punti per
cui $\alpha_i > 0$ sono detti *vettori di supporto (support vectors)* e
sono gli unici che contribuiscono alla definizione del vettore $w$. Essi
si trovano esattamente sul margine del classificatore e determinano
completamente la soluzione del problema.\
Dalle condizioni di complementarietà (KKT) si ricava che, per ogni
vettore di supporto $x_i$, vale: 
$$
y_i(w^\top x_i + b) = 1.
$$
Sostituendo l'espressione duale del vettore dei pesi
$$
w = \sum_{j=1}^{m} \alpha_j y_j x_j,
$$ 
si ottiene:
$$
y_i\left(\sum_{j=1}^{m} \alpha_j y_j \langle x_j, x_i \rangle + b\right) = 1.
$$
Da cui si ricava il valore del bias:
$$
b = y_i - \sum_{j=1}^{m} \alpha_j y_j \langle x_j, x_i \rangle,
\quad \forall i \in SV,
$$ 
dove $SV$ indica l'insieme dei vettori di
supporto. L'iperpiano di separazione assume quindi la forma:
$$
\pi(x) = w^\top x + b,
$$ 
con 
$$
w = \sum_{i=1}^{m} \alpha_i y_i x_i.
$$
La funzione di classificazione è, infine, definita come:
$$
h(x) = \mathrm{sgn}(\pi(x)).
$$

## SVM lineare - Caso non linearmente separabile

Pur essendo un modello teoricamente elegante (grazie alla formulazione
convessa e all'unicità di soluzione), l'SVM hard è fortemente limitato.
Infatti, l'assunzione di separabilità lineare raramente trova riscontro
nella realtà. Inoltre, è molto sensibile al rumore e agli outlier che
rendono il modello incapace di rappresentare la struttura globale dei
dati, dal momento che si adatta ai casi isolati.\
Per questi motivi, si preferisce rilassare l'ipotesi di separabilità
introducendo delle *variabili di slack* $\{\xi_i\}_{i = 1}^m$:
$$
y_i (w^\top x_i + b) \geq 1 - \xi_i, \quad i = 1, \dots, m.
$$ 
Il
modello risultante si chiama *soft margin SVM* e si pone l'obiettivo di
massimizzare il margine tenendo sotto controllo le \"violazioni\". Per
raggiungere questo scopo, si risolve il problema di ottimizzazione
vincolata: 
$$
\begin{equation*}
\begin{aligned}
 & \min_{w, b, \xi} && \frac{1}{2} \|w\|^2 + C \sum_{i = 1}^m \xi_i \\
 & \text{s.t.} && y_i(w^\top x_i + b) \ge  1 - \xi_i, \quad \forall i= 1, \dots, m, \quad \land \\
 & && \xi_i \ge 0, \quad \forall i = 1, \dots, m.
\end{aligned}
\end{equation*}
$$ 
Il *parametro di tradeoff* $C \ge 0$ controlla il
compromesso tra l'ampiezza del margine e la penalizzazione degli errori:

- valori elevati di $C$ penalizzano fortemente le violazioni, riducendo
  la tolleranza al rumore;

- valori piccoli di $C$ permettono maggiori violazioni, permettendo di
  migliorare la capacità di generalizzazione.

Anche in questo caso, il problema di ottimizzazione è convesso e ammette
un'unica soluzione globale.\
Introducendo i moltiplicatori di Lagrange
$\alpha_i, \beta_i \ge 0, \quad i = 1, \dots, m$, si ottiene che il
vettore dei pesi può essere scritto come:
$$
w = \sum_{i = 1}^m \alpha_i y_i x_i
$$ 
e che valgono i vincoli:
$$
\begin{equation*}
\begin{aligned}
& \sum_{i = 1}^m \alpha_i y_i = 0\\
& \alpha_i + \beta_i = C, \quad \forall i \in \{1, \dots, m\}.
\end{aligned}
\end{equation*}
$$ 
Le condizioni di compatibilità (KKT) sono:
$$
\begin{equation*}
\begin{aligned}
& \forall i \in \{1, \dots, m\}, \alpha_i [y_i(w^\top x_i + b) - 1 - \xi_i] = 0 \land  \beta_i \xi_i = 0
\end{aligned}
\end{equation*}
$$ 
Da esse si ricava che: 
$$
\alpha_i > 0
\quad \Longrightarrow \quad
y_i(w^\top x_i+b)=1-\xi_i.
$$ 
Inoltre, dal vincolo 
$$
\alpha_i+\beta_i=C
$$
segue che 
$$
0\leq \alpha_i \leq C.
$$ 
L'interpretazione geometrica delle
variabili duali è molto significativa:

- se $\alpha_i = 0$, il punto è correttamente classificato e si trova
  all'esterno del margine;

- se $0 < \alpha_i < C$, il punto giace esattamente sul bordo del
  margine;

- se $\alpha_i = C$, il punto si trova all'interno del margine oppure è
  classificato erroneamente.

I punti per i quali $\alpha_i > 0$ sono detti *vettori di supporto*
(*support vectors*). Analogamente al caso hard margin, essi sono gli
unici punti che contribuiscono alla determinazione del classificatore e
quindi del vettore dei pesi $w=\sum_{i=1}^{m}\alpha_i y_i x_i.$ Per
determinare il termine di bias $b$, si considera un vettore di supporto
tale che $0<\alpha_i<C.$ Per tali punti vale necessariamente
$\beta_i>0$ e dalla condizione di complementarità $\beta_i\xi_i=0$
si ottiene dunque $\xi_i=0$. Pertanto, si ricava:
$$
b = y_i - w^\top x_i 
=
y_i
-
\sum_{j=1}^{m}
\alpha_j y_j
\langle x_j,x_i\rangle,
\qquad
0<\alpha_i<C.
$$ 
Alla luce di questi risultati, la funzione di
classificazione assume, dunque, la forma: 
$$
h(x)
=
\operatorname{sgn}
\left(
\sum_{i=1}^{m}
\alpha_i y_i
\langle x_i,x\rangle
+b
\right).
$$

## Kernel SVM

La formulazione soft margin dell'SVM permette di gestire dataset non
perfettamente separabili, ma continua a ricercare una frontiera di
decisione lineare nello spazio originale dei dati. Tuttavia, in molte
applicazioni le classi non sono separabili mediante un iperpiano in
$\mathbb{R}^n$. Per questi motivi, si preferisce utilizzare le *Kernel
Support Vector Machine (Kernel SVM)* che si servono di una mappa non
lineare per mappare i dati in uno spazio di dimensione maggiore,
mediante la mappa non lineare:
$$
\phi: \mathbb{R}^n \rightarrow \mathcal{H},
$$ 
dove
$(\mathcal{H}, \langle \cdot, \cdot\rangle_\mathcal{H})$ è uno spazio di
Hilbert di dimensione più elevata rispetto allo spazio di partenza. In
tale spazio si gerca un classificatore lineare nella forma:
$$
h(x) = \operatorname{sgn}\bigl( w^\top\phi(x) + b \bigr) = \operatorname{sgn}\bigl( \sum_{i = 1}^m K(x, x_i) + b\bigr),
$$
dove
$K: \mathbb{R}^n \times \mathbb{R}^n \rightarrow \mathbb{R}, \quad K(x, z)  = \langle \phi(x), \phi(z) \rangle_\mathcal{H}$
prende il nome di *kernel*.\
Grazie a questa formulazione, è possibile lavorare implicitamente nello
spazio trasformato senza dover calcolare esplicitamente $\phi$. Tale
proprietà è nota come *kernel trick*.\
Inoltre, è possibile osservare che, anche in questo modello, il
classificatore dipende esclusivamente dai vettori di supporto, ovvero
dai dati $x_i$ per i quali vale $\alpha_i > 0$.\
Tra i kernel più utilizzati si annoverano:

- il *kernel lineare*, $K(x, z) = x^\top z$, utilizzato nell'SVM
  lineare;

- il *kernel polinomiale*: $K(x, z) = (x^\top z + c)^d$, con $c > 0$ e
  con $d$ grafo del polinomio; permette di modellare relazioni non
  lineari tra le variabili introducendo implicitamente termini
  quadratici, cubici e di ordine superiore;

- il *kernel gaussiano (RBF)*:
  $K(x, z) = \operatorname{exp}(-\gamma \| x- z\|^2), \quad \gamma = \frac{1}{2\sigma^2} >0$,
  in cui la similarità tra due punti decresce esponenzialmente con la
  loro distanza euclidea; se è piccolo, l'esponente decresce lentamente
  e anche punti relativamente lontani risultano ancora simili; ciò
  conduce a frontiere decisionali regolari e globali; per valori di
  $\gamma$ grandi, la similarità decresce molto rapidamente e solo punti
  molto vicini producono valori elevati nel kernel, permettendo di
  modellare strutture più complesse, ma aumentando il rischio di
  overfitting; inducendo uno spazio delle feature di dimensione
  infinita, l'RBF permette di generare frontiere decisionali molto
  flessibili; dal momento che è particolarmente efficace quando non si
  conosce la struttura dei dati, si tratta del kernel più utilizzato
  nella pratica;

- il *kernel sigmoide*:
  $K(x,z) = \tanh(\gamma x^\top z + c), \quad \gamma  > 0, \quad c \in \mathbb{R}^n$;
  il parametro $\gamma$ è un fattore di scala che determina quanto
  rapidamente il valore del kernel varia al variare della similarità tra
  i punti; al suo decrescere, il kernel tende a essere simile a uno
  lineare; per valori elevati di $\gamma$, la funzione sigmoide rende la
  frontiera decisionale più complessa e sensibile alle variazioni dei
  dati; la sua controparte nelle reti neurali è rappresentata dai pesi;
  il parametro $c$, invece, è analogo al bias nei neuroni.

Per eliminare l'influenza della norma dei vettori $x$ nello spazio delle
feature su $K$, si introducono i *kernel normalizzati (normalized
kernel)*: 
$$
\tilde K (x, z) = \begin{cases}
    0, \quad \text{se} \quad K(x,x) = 0 \quad \lor \quad K(z,z) = 0\\
    \frac{K(x,z)}{\sqrt{K(x,x)K(z,z)}}, \quad \text{altrimenti}
\end{cases}, \quad \forall x, z \in S, 
$$ 
dove $S$ è il training set.\
Così facendo, il kernel diventa una misura di similarità basata
esclusivamente sull'orientamento relativo delle osservazioni, più
facilmente interpretabile e numericamente più stabile.

## Classificatori multiclasse e SVM

Nonostante nascano come classificatori binari, è possibile estendere le
SVM ai problemi di classificazione multiclasse. Sono due le tecniche
principali: *One-vs-Rest (OvR)* e *One-vs-One (OvO)*.

### L'approccio One-vs-Rest (OvR)

L'approccio *One-vs-Rest (OvR)* estende le SVM al caso multiclasse
costruendo $K$ classificatori binari, dove $K$ è il numero delle classi.
Per ciascuna delle classi $k$, si addestra una SVM considerando come
esempi positivi tutti quelli appartenenti alla classe $k$ e come
negativi tutte le osservazioni appartenenti alle classi complementari.\
Con questo approccio, si ottengono $K$ funzioni decisionali
$f_1, \dots, f_K$, dove
$$
f_j(x) = \sum_{i = 1}^m \alpha_{i, j}y_iK(x_i, x) + b_j, \quad j = 1, \dots, K
$$
e con $m$ il numero di esempi nel training set $S$.\
Data una nuova osservazione $\hat x$, la classe predetta è quella
associata al valore più elevato della funzione decisionale:
$$
\hat y_{pred} = \arg \max_{k = 1, \dots, K} f_k(\hat x).
$$

### L'approccio One-vs-One (OvO)

L'approccio *One-vs-One (OvO)* estende le SVM al caso multiclasse
costruendo un classificatore binario per ogni coppia di classi. Dato un
problema con $K$ classi, si addestrano, dunque:
$$
\binom{K}{2} = \frac{K(K-1)}{2}
$$ 
SVM, ciascuna delle quali considera
i dati appartenenti a due classi $k_1$ e $k_2$.\
La predizione finale viene ottenuta mediante un meccanismo di votazione
maggioritaria: ciascuna classe assegna un \"voto\" alla classe da esso
predetta tra le due considerate; la nuova osservazione $\hat x$ viene
attribuita alla classe che raccoglie complessivamente il massimo numero
di voti: 
$$
\hat y_{pred} = \arg \max_{k = 1, \dots, K} \text{voti}_k.
$$

## SVM e VC Dimension

Un pilastro dello statistical learning è costituito dal concetto di *VC
dimension (dimensione di Vapnik-Chervonenskis)*, una misura della
capacità espressiva di una classe di modelli di classificazion. Data una
classe $\mathcal{H}$ di classificatori binari, la VC dimension di H
(indicata $VC-dim(\mathcal{H})$) è il massimo intero $d$ tale che esiste
un insieme di $d$ punti che viene *frantumato (shatterato)* da
$\mathcal{H}$.\
Nel caso di classificatori lineari, la VC dimension è legata alla
geometria del problema e, in particolare, al margine di separazione tra
classi. Detti $\rho$ il margine geometrico massimo ottenuto tra un
iperpiano separatore e $R$ il raggio della sfera contenente i dati, è
possibile dimostrare che la VC dimension soddisfa la relazione:
$$
\text{VC-dim} \;\lesssim\; \frac{R^2}{\rho^2}.
$$ 
Questo risultato
evidenzia come la capacità del modello sia legato in una relazione
inversa al margine di quest'ultimo: all'aumentare di $\rho$, la VC-dim
diminuisce, riducendo la complessità delle classi di ipotesi
considerata. Questo principio è sfruttato dalle SVM poichè la loro
formulazione consiste nel massimizzare il margine tra le classi. In tal
senso, il problema di ottimizzazione delle SVM non si limita alla
ricerca di un iperpiano separatore, ma realizza implicitamente un
controllo della capacità del modello. Di conseguenza, la massimizzazione
del margine può essere interpretata come una forma di regolarizzazione
geometrica, capace di ridurre la VC-dim e di migliorare la capacità di
generalizzazione del classificatore sui dati non osservati.




## SVM e GAMMA Magic Telescope
Dal momento che le proiezioni FDA e PCA hanno evidenziato una sovrapposizione non linearmente separabile tra le classi, addestriamo prevalentemente una support vector machine (SVM) con kernel non lineare. Tuttavia, per scopi puramente dimostrativi, proviamo ad effettuare una SVM lineare sui dati ridotti a 2 PC.


### SVM Lineare sui dati ridotti a 2 PC: Hard&Soft
L'obiettivo di questa sezione è visualizzare l'effetto dell'iperparametro C dopo aver proiettato i dati sulle prime due componenti principali (per facilitare la visualizzazione). Si confrontano un margine duro (hard margin, *C = 1000*) e un margine morbido (soft margin, *C=0.001*). <br> Questo esperimento ha valore puramente illustrativo: la riduzione a 2 PC 
comporta una perdita di informazione significativa, e la SVM lineare 
conferma quanto già suggerito dalla FDA — le due classi non sono 
linearmente separabili. Questo motiva il ricorso al kernel RBF nella 
sezione successiva.

In [ ]:
#riduzione della dimensionalità --> 2 pc
pca2 = PCA(n_components=2, random_state=42)
X_trainval_2d = pca2.fit_transform(X_trainval_scaled_tv)
X_test_2d     = pca2.transform(X_test_scaled_tv)
#hard SVM (C alto --> margine stretto, pochi errori tollerati)
lsvm_h = LinearSVC(C=1000, loss="squared_hinge", dual=False,
                   class_weight="balanced", max_iter=10000, random_state=random_state)
lsvm_h.fit(X_trainval_2d, y_trainval)
#soft SVM (C basso --> margine largo, più errori tollerati)
lsvm_s = LinearSVC(C=0.01, loss="hinge",         dual=True,
                   class_weight="balanced", max_iter=10000, random_state=random_state)
lsvm_s.fit(X_trainval_2d, y_trainval)
def make_line_xy(w, b, x_range):
    #Restituisce iperpiano e i due margini nel piano PC1-PC2.
    #Iperpiano: w0*x + w1*y + b = 0  →  y = -(w0*x + b) / w1
    xs      = np.array(x_range)
    ys      = -(w[0]*xs + b)     / w[1]
    ys_top  = -(w[0]*xs + b - 1) / w[1]   # margine +1
    ys_bot  = -(w[0]*xs + b + 1) / w[1]   # margine -1
    return xs, ys, ys_top, ys_bot
x_range = [X_trainval_2d[:, 0].min()-0.5, X_trainval_2d[:, 0].max()+0.5]
xs_h, ys_h, yt_h, yb_h = make_line_xy(lsvm_h.coef_[0], lsvm_h.intercept_[0], x_range)
xs_s, ys_s, yt_s, yb_s = make_line_xy(lsvm_s.coef_[0], lsvm_s.intercept_[0], x_range)

fig = make_subplots(rows=1, cols=2, subplot_titles=(
    f"Hard SVM (C=1000) — acc train+val: {lsvm_h.score(X_trainval_2d, y_trainval)*100:.1f}%",
    f"Soft SVM (C=0.01)  — acc train+val: {lsvm_s.score(X_trainval_2d, y_trainval)*100:.1f}%",
))
color_map = {0: "#E63946", 1: "#2E86AB"}
label_map  = {0: "gamma (g)", 1: "hadron (h)"}

for col, (xs, ys, yt, yb) in enumerate(
        [(xs_h, ys_h, yt_h, yb_h), (xs_s, ys_s, yt_s, yb_s)], start=1):
    for cls in [0, 1]:
        mask = y_trainval == cls
        fig.add_trace(go.Scatter(
            x=X_trainval_2d[mask, 0], y=X_trainval_2d[mask, 1],
            mode="markers", name=label_map[cls], legendgroup=label_map[cls],
            marker=dict(color=color_map[cls], size=5, opacity=0.5,
                        line=dict(width=0.3, color="DarkSlateGrey")),
            showlegend=(col == 1),
        ), row=1, col=col)
    # iperpiano e margini
    fig.add_trace(go.Scatter(x=xs, y=ys, mode="lines",
                             line=dict(color="black", width=2),
                             showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=xs, y=yt, mode="lines",
                             line=dict(color="black", dash="dash", width=1),
                             showlegend=False), row=1, col=col)
    fig.add_trace(go.Scatter(x=xs, y=yb, mode="lines",
                             line=dict(color="black", dash="dash", width=1),
                             showlegend=False), row=1, col=col)
    fig.update_xaxes(title_text="PC1", row=1, col=col,
                     range=[X_trainval_2d[:, 0].min()-0.5, X_trainval_2d[:, 0].max()+0.5])
    fig.update_yaxes(title_text="PC2", row=1, col=col,
                     range=[X_trainval_2d[:, 1].min()-0.5, X_trainval_2d[:, 1].max()+0.5])

fig.update_layout(height=450, width=1100,
                  title_text="Confronto Hard vs Soft SVM lineare — dati ridotti a 2 PC")
fig.show()
auc_lsvm_h = roc_auc_score(y_test, lsvm_h.decision_function(X_test_2d))
auc_lsvm_s = roc_auc_score(y_test, lsvm_s.decision_function(X_test_2d))
print(f"AUC Hard SVM (C=1000) — test set: {auc_lsvm_h:.4f}")
print(f"AUC Soft SVM (C=0.01) — test set: {auc_lsvm_s:.4f}")

Questa visualizzazione conferma quanto già suggerito dalle proiezioni PCA e FDA: il confine di decisione tra gamma e hadron non è linearmente separabile nello spazio delle prime due componenti principali. In entrambi gli iperpiani (hard e soft SVM) non è possibile separare le due classi in maniera soddisfacente, poichè le distribuzioni si sovrappongono estesamente lungo entrambe le direzioni. <br> La differenza di AUC tra Hard e Soft SVM è trascurabile perché il problema non è il valore di C, ma il fatto che un confine lineare è intrinsecamente inadeguato per questo dataset. Ridurre ulteriormente C non migliora la generalizzazione quando il bias del modello è troppo alto — siamo in una situazione di underfitting strutturale, non di overfitting. <br> In generale, con valori grandi di C, il margine è stretto e il modello cerca di minimizzare le violazioni, adattandosi maggiormente ai dati di training e rischiando così l'overfitting; con C basso, invece, il margine è largo e il modello è più tollerante nei confronti degli errori di classificazione sul training set, privilegiando la generalizzazione. <br> Lavorando con il dataset MAGIC Gamma Telescope, tuttavia, la separazione lineare risulta insufficiente, motivando il ricorso a un kernel non lineare. 

### Kernel SVM con Grid Search degli iperparametri
Per la validazione, si utilizza il validation set predefinito invece di una k-fold CV: si passa a GridSearchCV un iteratore che genera una sola coppia *(train_idx, val_idx)*. <br>
Ci si serve di una concatenazione di *X_trainval_scaled* e di *y_trainval* che per costruzione contiene prima i valori relativi al training set e, successivamente, quelli relativi al validation set. Di conseguenza, gli indici *np.arange(n_tr)* puntano al training set e *np.arange(n_tr, n_tr + len(y_val))* al validation set. Se il dataset avesse avuto poche osservazioni, si sarebbe potuta utilizzare una procedura di *k-fold cross-validation*, in cui il training set viene suddiviso in $k$ sottoinsiemi e a turno ciascuno di essi viene usato per la validazione, mentre i rimanenti per l'addestramento. Le prestazioni del modello, quindi, sono stimate servendosi della media aritmetica delle metriche ottenute sui diversi fold. La scelta di utilizzare un validation set fisso, dettata dalla numerosità del MAGIC Gamma Dataset, permette di ridurre il costo computazionale e di usare lo stesso insieme di validazione per un confronto uniforme dei diversi modelli considerati.

In [ ]:
#in questo caso il validation set è utilizzato come tale
weight = 'balanced'
X_combined = np.concatenate([X_train_scaled, X_val_scaled], axis = 0)
y_combined = np.concatenate([y_train, y_val], axis = 0)
#effettuo una grid search esplorativa sul subsample
n_tr     = len(y_train)
cv_split = [(np.arange(n_tr), np.arange(n_tr, n_tr + len(y_val)))] #così utilizzo il training e il validation set costruiti in precedenza
#in X_scaled gli elementi del validation e del training set sono mescolati randomicamente,
#con cv_split considero i primi n_tr elementi come sample del training set, gli ultimi come sample del validation set
C_list     = [2**i for i in range(-3,3)]
gamma_list = [1 / (k * X_train_scaled.shape[1]) for k in np.arange(0.5, 2.75, 0.5)]
ker_list   = ["rbf", "sigmoid", "poly"]
hparameters = [
    {"C": C_list, "gamma": gamma_list, "kernel": ker_list},  
]
svm_gs = SVC(class_weight= weight, cache_size=400, random_state=random_state)
gs = GridSearchCV(
    estimator=svm_gs, param_grid=hparameters,
    scoring="roc_auc", cv=cv_split,
    n_jobs=-1, refit=True, verbose=0, return_train_score=True,
)
#cv = cv_split forza l'uso di uno split fisso definito dall'utente; se avessi usato cv = 5 avrei avuto cross validation con 5 k-fold, ovvero suddividendo il 
#dataset in 5 set di cui uno è usato a turno per la validazione
#usando GridSearchCV con cv_split 
gs.fit(X_combined, y_combined)
print(f"Miglior configurazione : {gs.best_params_}")
print(f"Miglior AUC validation : {gs.best_score_:.4f}")
#top 5 configurazioni
df_results = pd.DataFrame(gs.cv_results_)
top5 = (
    df_results
    .sort_values("rank_test_score")
    .head(5)[[
        "param_kernel", "param_C", "param_gamma",
        "mean_test_score", "mean_train_score", "rank_test_score",
    ]]
    .rename(columns={
        "param_kernel":     "Kernel",
        "param_C":          "C",
        "param_gamma":      "γ",
        "mean_test_score":  "AUC val",
        "mean_train_score": "AUC train",
        "rank_test_score":  "Rank",
    })
)
top5.style.format({
    "AUC val":   "{:.4f}",
    "AUC train": "{:.4f}",
    "C":         "{}",
    "γ":         "{}",
}).hide(axis="index")



Così facendo, si ricava la migliore configurazione sulla quale effettuare l'SVM. Si osserva, inoltre, che il kernel RBF è il pi competitivo tra quelli esplorati: la sua capacità di modellare confini di decisione non lineari nello spazio delle feature si adatta bene alla struttura dei dati, che si è già rilevata essere non linearmente separabile. <br> Il parametro $\gamma = 0.1$ governa il raggio di influenza di ciacun vettore di supporto.  Il valore $C = 4$ (moderatamente piccolo) indica che si preferisce un soft margin: il modello tollera alcune violazioni del margine al fine di migliorare la generalizzazione. <br> La GridSearch con *refit = True* ha già rifittato il miglior modello su *X_trainval_scaled_tv*. Tuttavia, per ottenere *predict_proba*, occorre rifittare con *probability = True*.

In [38]:
best = gs.best_params_
svm_best = SVC(
    class_weight=weight,
    kernel=best["kernel"],
    C=best["C"],
    gamma=best.get("gamma", "scale"), 
    probability=True,
    random_state=42,
)
# Il modello finale è riaddestrato su train+val con scaler rifittato su train+val.
# Il test set e' trasformato con lo stesso scaler train+val: nessuna statistica del test entra nel fit.
svm_best.fit(X_trainval_scaled_tv, y_trainval)
y_train_pred_svm  = svm_best.predict(X_train_scaled_tv)
y_train_proba_svm = svm_best.predict_proba(X_train_scaled_tv)[:, 1]
y_val_pred_svm    = svm_best.predict(X_val_scaled_tv)
y_val_proba_svm   = svm_best.predict_proba(X_val_scaled_tv)[:, 1]
y_test_pred_svm   = svm_best.predict(X_test_scaled_tv)
y_test_proba_svm  = svm_best.predict_proba(X_test_scaled_tv)[:, 1]
# dopo il refit su train+val, l'AUC su y_val sarebbe già stata "vista"
# si riporta l'AUC della grid search.
auc_svm_val  = gs.best_score_          #non fa parte del sample
auc_svm_test = roc_auc_score(y_test, y_test_proba_svm)
print(f"AUC SVM (val, selezione) : {auc_svm_val:.4f}")
print(f"AUC SVM test             : {auc_svm_test:.4f}")

AUC SVM (val, selezione) : 0.9229
AUC SVM test             : 0.9243


Nel modello finale si osservano valori coerenti tra test e validation set.

Si realizzano ora le curve ROC per confrontare la SVM con la QDA sul test set.

In [ ]:
fpr_s, tpr_s, _ = roc_curve(y_test, svm_best.predict_proba(X_test_scaled_tv)[:, 0], pos_label=0)
#recupero queste informazioni sulla qda dalla sezione apposita
plt.figure(figsize=(7, 6))
plt.plot(fpr_s, tpr_s, label=f"SVM {best['kernel'].upper()} (AUC = {auc_svm_test:.4f})",
         color='steelblue')
plt.plot(fpr_qda_t, tpr_qda_t, label=f"QDA (AUC = {auc_qda_t:.4f})",
         color='tomato', linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — SVM vs QDA (test set)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

La curva ROC conferma la superiortà della SVM RBF rispetto alla QDA sul test set. La curva dell'SVM si mantiene, infatti, sistematicamente più alta di quella della QDA lungo tutto il range di soglie. Ciò significa che, a parità di false positive rate accettato, la SVM garantisce un true positive rate maggiore. Nel contesto del MAGIC Gamma Dataset, questo dettaglio è particolarmente rilevante: come indicato nella documentazione ufficiale, la probabilità di accettare un evento andronico come segnale deve essere mantenuta sotto soglie molto stringenti.

In conclusione, visualizziamo la performance della miglior SVM individuata con la gridsearch:
1. Accuratezza su training,validation e test set
2. Precisione su training,validation e test set
3. Recall su training,validation e test set
4. AUC su training, validation e test set
Realizziamo, inoltre, le matrici di confusione sul test set

In [ ]:
class_names = ['gamma (g)', 'hadron (h)']
df_performance = pd.DataFrame({
    'Accuracy':  [accuracy_score(y_train, y_train_pred_svm),
                  accuracy_score(y_val,   y_val_pred_svm),
                  accuracy_score(y_test,  y_test_pred_svm)],
    'Precision': [precision_score(y_train, y_train_pred_svm, average='weighted'),
                  precision_score(y_val,   y_val_pred_svm,   average='weighted'),
                  precision_score(y_test,  y_test_pred_svm,  average='weighted')],
    'Recall':    [recall_score(y_train, y_train_pred_svm, average='weighted'),
                  recall_score(y_val,   y_val_pred_svm,   average='weighted'),
                  recall_score(y_test,  y_test_pred_svm,  average='weighted')],
    'AUC':       [roc_auc_score(y_train, y_train_proba_svm),
                  roc_auc_score(y_val,   y_val_proba_svm),
                  roc_auc_score(y_test,  y_test_proba_svm)]
    }, index=['training', 'validation', 'test'])
display(df_performance.round(4))
#matrici di confusione
cmat = confusion_matrix(y_test.astype(int), y_test_pred_svm, labels=svm_best.classes_)
cmat_norm_true = confusion_matrix(y_test.astype(int), y_test_pred_svm, labels=svm_best.classes_, normalize='true')  #recall_confusion_matrix
cmat_norm_pred = confusion_matrix(y_test.astype(int), y_test_pred_svm, labels=svm_best.classes_, normalize='pred')  #precision_confusion_matrix
#dataframes
df_cmat = pd.DataFrame(cmat, columns=class_names, index=class_names)
df_cmat_norm_true = pd.DataFrame(cmat_norm_true, columns=class_names, index=class_names)
df_cmat_norm_pred = pd.DataFrame(cmat_norm_pred, columns=class_names, index=class_names)
print("\nMatrice di confusione (valori assoluti):")
display(df_cmat)
print("\nMatrice di confusione (normalizzata rispetto alle vere classi - recall):")
display(df_cmat_norm_true.round(4))
print("\nMatrice di confusione (normalizzata rispetto alle classi predette - precision):")
display(df_cmat_norm_pred.round(4))
#visualizzazione grafica
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(cmat, display_labels=class_names).plot( ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title("Valori assoluti")
ConfusionMatrixDisplay(cmat_norm_true, display_labels=class_names).plot( ax=axes[1], colorbar=False, cmap='Blues', values_format='.3f')
axes[1].set_title("Normalizzata sulle vere classi (recall)")
ConfusionMatrixDisplay(cmat_norm_pred, display_labels=class_names).plot( ax=axes[2], colorbar=False, cmap='Blues', values_format='.3f')
axes[2].set_title("Normalizzata sulle classi predette (precision)")
plt.suptitle(f"Matrici di confusione — SVM {best['kernel'].upper()} (test set)", fontsize=13)
plt.tight_layout()
plt.show()

La tabella delle performance mostra che i tre set sono coerenti per tutte le metriche: si ottengono, infatti, valori quasi identici tra il training, il validation e il test set; ciò è indicatore dell'assenza di overfitting. In particolare, la coerenza dell'AUC suggerisce che il modello ha appreso pattern generalizzabili e non ha memorizzato il training set. <br>
Le matrici di confusione normalizzate rivelano un comportamento asimmetric tra le due classi, riflesso dello sbilanciamento del dataset. La matrice normalizzata sulle vere classi (recall) mostra che la classe hadron viene riconosciuta in misura inferiore rispetto alla classe gamma (e ciò è atteso per via dello sbilanciamento del dataset). L'uso di *class_weight = 'balanced'* mitiga parzialmente questo aspetto. La matrice normalizzata rispetto alle classi predette (precision) indica, invece, la percentuale di predizioni corrette per ogni classe.

Realizziamo una tabella riassuntiva finale:

In [ ]:
auc_lda_test = roc_auc_score(y_test, lda.predict_proba(X_test_scaled)[:,1])
auc_lda_v = roc_auc_score(y_val, lda.predict_proba(X_val_scaled)[:,1])
auc_qda_v = roc_auc_score(y_val, qda.predict_proba(X_val_scaled)[:,1])
summary = pd.DataFrame({
    'Modello':        ['LDA', 'QDA', f"SVM {best['kernel'].upper()}"],
    'AUC Validation': [auc_lda_v, auc_qda_v, auc_svm_val],
    'AUC Test':       [auc_lda_test, auc_qda_t, auc_svm_test]
})
print("RIEPILOGO:")
print(summary.to_string(index=False))

La tabella riassuntiva finale mostra che la SVM RBF supera nettamente sia l'LDA sia la QDA sul test set. Ciò non è inaspettato: la SVM con kernel RBF non è vincolta da nessuna assunzione sulla distribuzione dei dati ed è quindi capace di modellare confini di decisione arbitrariamente complessi. Al contrario, le ipotesi alla base dell'LDA e della QDA (normalità e, rispettivamente, omoschedasticità ed eteroschedasticità delle distribuzioni delle classi) non sono pienamente soddisfatte da questo dataset.

# Classificazione: Multi-Layer Perceptron (MLP)

Il *Multi-Layer Perceptron (Percettrone Multi-Stato, MLP)* è una rete
neurale (neural network) feed forward composta da più strati di neuroni
tra loro interconnessi. Si tratta di una generalizzazione del
percettrone semplice e di uno dei modelli fondamentali del supervised
learning.\
L'idea teorica alla base dei multilayer perceptron può essere ricondotta
ai risultati di Kolmogorov,
secondo cui funzioni continue complesse possono essere espresse
mediante composizioni di funzioni più semplici.\
Nell'MLP si vuole, infatti, apprendere una funzione
$f: \mathbb{R}^n \rightarrow \mathbf{R}^m$ capace di associare input e
output tramite una composizione di trasformazioni affini e funzioni non
lineari. Un MLP è composto da diversi \"strati\" (layer) di neuroni:

- input layer;

- hidden layers; se l'MLP presenta un solo hidden layer, è detta essere
  *shallow*; altrimenti, si tratta di una *deep neural network*; il loro
  numero determina la *profondità (depth)* della rete;

- output layer.

Consideriamo un MLP con $N \in \mathbb{N}$ layer. L'*input layer*
\"riceve\" il vettore delle feature
$x = (x_1, \dots, x_n)^\top \in \mathbb{R}^n$. A ogni componente di $x$
(*neurone*) corrisponde una variabile del dataset. Lo strato di input
non effettua calcoli, ma trasmette i dati allo strato successivo.\
Ogni neurone di un layer è connesso ai vettori del layer successivo
tramite dei pesi. Detti:

- $a_i^{(l)}$ *l' attivazione dell'i-esimo layer $l$*,

- $w_{ji}^{(l)}$ il peso tra il neurone $i$ del layer $l$ e il neurone
  $j$ del layer $l+1$,

allora il neurone successivo calcola la *preattivazione del $l+1$-esimo
layer*: 
$$
z_j^{(l+1)} = \sum_i w_{ji}^{(l)}a_j^{(l)} + b_j^{(l)},
$$ 
con
$b_j^{(l)}$ bias e $z_j^{(l+1)}$ è la combinazione lineare in ingresso
nella funzione non lineare. Raccogliendo i pesi in una matrice $W^{(l)}$,
le attivazioni e i bias rispettivamente nei vettori $b^{(l)}$ e
$a^{(l)}$, è possibile riscrivere la mappa affine in forma matriciale:
$$
z^{(l+1)} = W^{(l)}a^{(l)} + b^{(l)}.
$$ 
Dopo aver applicato la
trasformazione affine, il neurone applica una funzione lineare:
$$
a^{(l+1)} = \sigma(z^{(l+1)}).
$$ 
La non linearità della funzione di
attivazione $\sigma$ è fondamentale in quanto altrimenti l'intera rete
sarebbe equivalente a una singola trasformazione affine.\
Tra le funzioni di attivazione più comuni rientrano:

- *funzione di attivazione sigmoide o logistica*
  $\sigma(x) = \frac{1}{1 + \exp(-x)} \in (0,1)$; essa è liscia e vale
  $\sigma'(x) = \sigma(x)\sigma(-x)$; tuttavia, per
  $|x| \rightarrow + \infty$, può incorrere in problemi di *scomparsa
  del gradiente*;

- *funzione di attivazione tangente iperbolica*
  $\sigma(x) = \tanh(x) \in (-1,1)$ il cui gradiente solitamente produce
  valori più elevati di quello della sigmoide, ma che può, comunque,
  incorrere nel problema di scomparsa del gradiente;

- *funzione di attivazione Rectified Linear Unit (ReLU)*
  $\sigma(x) = \max(0,1)$, molto utilizzata nelle reti profonde per la
  sua semplicità computazionale e la stabilità numerica; può, tutavia,
  soffrire del problema della \"morte dei neuroni\".

Gli *hidden layers* permettono alla rete di apprendere rappresentazioni
sempre più astratte e complesse dei dati. In sintesi, ogni layer
applica: 
$$
a^{(l+1)} = \sigma(W^{(l)}a^{(l)} + b^{(l)}).
$$ 
Componendo
molte trasformazioni non lineari, è possibile approssimare funzioni
molto complesse.\
L'*output layer (L)* produce la predizione finale, trasformando i valori
numerici prodotti dalla rete in un output interpretabile rispetto al
problema che si desidera risolvere. Nel caso dei problemi di
regressione, non ha senso limitare artificialmente l'output della rete
in un intervallo prefissato; si applica quindi la funzione identità alla
pre-attivazione $z^{(L)}$.\
Nella classificazione binaria, si applica la funzione sigmoide alla
pre-attivazione $z^{(L)}$, in quanto l'output deve rappresentare una
probabilità $P(Y = 1 \mid X = x) \in [0,1]$. Tipicamente, se
$\sigma(z^{(L)}) > 0.5$ si assegna la classe $1$, altrimenti la classe
$0$.\
Per quanto riguarda i problemi di classificazione multiclasse con
insieme di classi $\mathcal{K} = \{1, \dots, K\}$, si utilizza la
*funzione softmax*: $$\hat y_i =
\frac{\exp\left(z_i^{(L)}\right)}
{\sum_{j=1}^{K} \exp\left(z_j^{(L)}\right)},
\qquad i = 1, \dots, K,$$ che può essere interpretata come la
probabilità di appartenenza alla classe $i$.\
L'osservazione $x$ viene assegnata alla classe che massimizza tale
probabilità, ovvero: $$\hat y = \arg\max_{i \in \mathcal{K}} \hat y_i.$$
Il processo di propagazione in avanti (feed-forward propagation)
consiste quindi in:
$$x \rightarrow z^{(1)} \rightarrow a^{(1)} \rightarrow z^{(2)} \rightarrow a^{(2)} \rightarrow \dots \rightarrow z^{(L)} \rightarrow \hat y.$$
Più rigorosamente, indicando con $M^{(l)}(a^{(l-1)}) =
W^{(l)}a^{(l-1)} + b^{(l)}$ la trasformazione lineare associata al layer
$l$, e con $\sigma^{(l)}$ la funzione di attivazione degli hidden layer
e con $\Phi$ la funzione di attivazione associata all'output layer, si
può descrivere il multi-layer perceptron come composizione di funzioni.
Complessivamente, infatti, realizza la funzione $
f :
\mathbb{R}^n \to \mathbb{R}^m$ definita da 
$$
f(x)
=
\Phi
\circ
M^{(L)}
\circ
\sigma^{(L-1)}
\circ
M^{(L-1)}
\circ
\dots
\circ
\sigma^{(1)}
\circ
M^{(1)}(x).
$$
 La qualità dell'approssimazione prodotta dalla rete
neurale viene misurata tramite la *loss function* che si pone
l'obiettivo di quantificare la distanza tra l'output previsto dalla rete
e il valore realmente associato ai dati di training. Indicando con $y$
il valore reale e con $\hat y$ il valore prodotto dalla rete, la loss
function associa a ogni osservazione un errore numerico. L'addestramento
mira a determinare i parametri[^4] della rete (pesi e bias) che
minimizzano l'errore totale sull'intero dataset su cui avviene il
training.\
La scelta della funzione di loss dipende dal tipo di problema
considerato. Nella regressione, una delle funzioni più utilizzate è il
*mean square error (MSE, errore quadratico medio)*:
$$
\mathcal{L}(y, \hat y) = \frac{1}{|\mathcal{T}|} \sum_{i = 1}^{|\mathcal{T}|} (y_i - \hat y_i)^2,$$
dove $\mathcal{T} = {(x_i, y_i)}_{i = 1}^{|\mathcal{T}|}$ è il training
set. Tale funzione attua una penalizzazione quadratica degli errori, in
maniera tale che predizioni molto distanti dal valore reale
contribuiscano in maniera più significativa alla loss complessiva.\
Un' alternativa che penalizza maggiormente predizioni molto distanti dal
valore reale è la penalizzazione *mean absolute error (media di errori
assoluti (MAE)*:
$$
\mathcal{L} (y, \hat y) = \frac{1}{|\mathcal{T}|} \sum_{i = 1}^{|\mathcal T|} |y_i - \hat y_i|,
$$
dove $\mathcal{T} = {(x_i, y_i)}_{i = 1}^{|\mathcal{T}|}$ è il training
set.\
Nei problemi di classificazione la funzione di loss più utilizzata è la
*0-1 loss* che assegna valore unitario alle classificazioni corrette e
nullo altrimenti:
$$
\mathcal{L} (y, \hat y) = \sum_{i = 1}^{|\mathcal{T}|} \mathbb{I}_{\hat y_i \neq y},
$$
dove $\mathcal{T} = {(x_i, y_i)}_{i = 1}^{|\mathcal{T}|}$ è il training
set e $\mathbb{I}$ è la funzione indicatrice. Minimizzare la 0-1 loss
equivale, dunque, a minimizzare direttamente il numero di errori di
classificazioni. Tuttavia, essa non fornisce informazioni \"graduali\"
sull'errore: per esempio, probabilità $0.001$ e $0.499$ di appartenere a
una classe forniscono entrambe errore unitario.\
Nei problemi di classificazione multiclasse si utilizza frequentemente
la *cross-entropy loss*. Essa presenta la forma:
$$
\mathcal{L} (y, \hat y) = -\sum_{i = 1}^{K} y_i \log (\hat y_i),
$$
dove

- $K$ rappresenta il numero delle classi;

- $y_i$ è la vera distribuzione di probabilità associata alla classi
  $i$;

- $\hat y_i$ è la probabilità predetta dalla rete tramite la funzione
  softmax.

La cross-entropy penalizza fortemente le situazioni in cui la rete
assegna bassa probabilità alla classe corretta.\
Scelta la loss da impiegare, l'obiettivo del training consiste nel
determinare i parametri della rete neurale (pesi e bias) che minimizzano
tal funzione sull'insieme di training:
$$
\arg\min_{w} \mathcal{L}(y,\hat y) = \arg \min_w \mathcal{L}(y, f_w(x)),
$$
dove $w$ è il vettore in cui sono concatenate le matrici dei pesi
$W^{(h)}$ e i bias $b^{(i)}$ della rete. Si tratta di un problema
generalmente ad alta dimensione la cui soluzione viene trovata
servendosi di metodi iterativi di ottimizzazione numerica, il più
utilizzato dei quali è la *discesa del gradiente (gradient descent).*
L'idea alla base consiste nell'aggiornamento iterativo dei parametri di
massima decrescita della funzione di loss. Indicando con $\alpha_k > 0$
il *learning rate*, la regola di aggiornamento è data da:
$$
w \leftarrow w - \alpha \nabla_w \mathcal{L}(y, f_w(x)).
$$ 
Il calcolo
del gradiente $\nabla_w \mathcal{L}(y, f_w(x))$, tuttavia, non è
immediato, ma è ottenuto applicando ripetutamente la *chain rule (regola
della catena)* alla funzione ottenuta come composizione della loss con
le trasformazioni non lineari corrispondenti ai layer della rete
neurale. Tale procedura è alla base dell'algoritmo di *backpropagation*,
che permette di calcolare in modo efficiente i gradienti propagando
l'errore dall'output verso i layer precedenti.\
Tuttavia, nella pratica si usa il *Stochastic (o mini-batch) Gradient
Descent (SGD)*, una variante del gradient descent che, ad ogni
iterazione, calcola il gradiente della funzione di loss su un
sottoinsieme (*mini-batch*) del training set, invece che sull'intero
dataset. Indicando con $B \subset \{1,\dots,m\}$ un mini-batch,
l'aggiornamento dei parametri è dato da:
$$
w \leftarrow w - \alpha \nabla_w \frac{1}{|B|}\sum_{i \in B} \mathcal{L}(y^{(i)}, f_w(x^{(i)})),
$$
con $\mathcal{L}$ una funzione di loss.\
Dal punto di vista algoritmico, tale aggiornamento  non si limita a un singolo sottoinsieme, ma viene esteso sistematicamente a tutta la base dati:
1. Partizione del dataset: all'inizio di ogni ciclo, l'intero training set composto da $m$ istanze viene rimescolato casualmente (*shuffling*) e suddiviso in un numero finito di mini-batch disgiunti $\{B_1, B_2, \dots, B_k\}$, ciascuno di cardinalità pari alla dimensione del batch stabilita ($|B|$).
2. Ciclo delle iterazioni (step): l'algoritmo esegue sequenzialmente la formula di aggiornamento dei parametri per ciascun mini-batch. Ad ogni step $j$, il gradiente viene stimato esclusivamente sulle informazioni contenute nel batch $B_j$, modificando immediatamente il vettore dei pesi $w$ prima di passare al batch successivo.
3. Un'epoca  si definisce completata nel momento esatto in cui l'algoritmo ha processato sequenzialmente tutti i $k$ mini-batch, esaurendo l'intero dataset di addestramento. Il processo complessivo viene tipicamente reiterato per un numero elevato di epoche fino al raggiungimento dei criteri di convergenza stabiliti.
  
 Nel mini-batch gradient descent, quindi, il gradiente della funzione
obiettivo viene approssimato calcolando il gradiente della media della
funzione di loss sul sottoinsieme di dati considerato, ottenendo così
una stima computazionalmente efficiente del gradiente globale.
Introducendo una lieve stocasticità, inoltre, si favorisce la
convergenza e si migliora la capacità di generalizzazione del modello. \
L'addestramento della rete viene arrestato secondo opportuni *criteri di
arresto*. Uno di essi consiste nel fissare un numero massimo di
*epoche*, ovvero di cicli completi di addestramento in cui la rete
neurale elabora una volta tutti gli esempi del training set
(eventualmente divisi in mini-batch). Decisamente più efficace è
l'approccio dell'*early stopping*, che interrompe il training quando il
validation score (accuracy sul validation set) smette di migliorare per
un certo numero di iterazioni consecutive (patience). Questa tecnica
consente di ridurre il rischio di overfitting, migliorando la capacità
di generalizzazione del modello.

## L'Adam Optimizer 

Nell'addestramento dell'MLP e, più in generale, delle DNN, un algoritmo
di ottimizzazione stocastica molto utilizzato è l'*Adam optimizer
(Adaptive Moment Estimation)*. Esso combina le idee del *momentum*
(ovvero la memoria della direzione dei gradienti) e dell'adattamento del
learning rate con l'obiettivo di migliorare la stabilità e la velocità
di convergenza del metodo di discesa del gradiente.\
Sia $g(t) = \nabla_w \mathcal{L}(t)$ il gradiente della loss function al
tempo $t$. Adam costruisce due stime esponenzialmente pesate:
$$
m_t = \beta_1 m_{t-1} + (1-\beta_1) g_t,
\qquad
v_t = \beta_2 v_{t-1} + (1-\beta_2) g_t^2,
$$ 
che rappresentano
rispettivamente una media dei gradienti e una media dei quadrati dei
gradienti. I parametri $\beta_1$ e $\beta_2$ sono *coefficienti di
decadimento esponenziale (exponential decay rates)* che controllano
quanta \"memoria\" del passato viene mantenuta nelle medie dei
gradienti. In particolare, $\beta_1$ controlla la media dei gradienti
(momentum) determinando quanto pesano i gradienti passati rispetto a
quello attuale. Più è prossimo ad 1, più è forte la memoria del passato
e il movimento è stabile, ma lento; più è prossimo a 0, più la direzione
segue rapidamente il gradiente corrente. Tipicamente
$\beta \approx 0.9$. Il parametro $\beta_2$ controlla la media dei
quadrati dei gradienti e serve per stimare la varianza del gradiente
stesso. Più il valore di $\beta_2$ è prossimo a 1, più è stabile la
stima della scala dei gradienti, più è prossimo a 0, più è sensibile la
risposta alle variazioni recenti. Tipicamente $\beta_2 \approx 0.999$.\
Per correggere il bias introdotto dall'inizializzazione nulla, si
definiscono le quantità corrette: 
$$
\hat m_t = \frac{m_t}{1-\beta_1^t},
\qquad
\hat v_t = \frac{v_t}{1-\beta_2^t}.
$$ 
L'aggiornamento dei parametri è,
dunque, dato da: 
$$
w_{t+1}
=
w_t
-
\alpha
\frac{\hat m_t}{\sqrt{\hat v_t} + \varepsilon},$$ dove $\alpha$ è il
learning rate e $\varepsilon > 0$ è una costante numerica introdotta per
garantire la stabilità.\
Così facendo, Adam combina la direzione media dei gradienti con una
normalizzazione adattativa della loro varianza, permettendo
aggiornamento più stabili e robusti rispetto al gradient descent
classico, soprattutto nel caso di problemi in spazi di grandi
dimensioni.

### MLP & MAGIC Gamma Telescope: dati trattati con PCA, rete neurale shallow
Per prima cosa, viene addestrata una rete MLP shallow, ovvero composta da un solo hidden layer di cinquanta neuroni, sui dati ridotti tramite PCA. Si utilizzano 50 neuroni e la funzione di attivazione logistica (sigmoide).<br> L'obiettivo è verificare se una rete poco pronda è capace di cogliere la struttura non lineare dei dati, prima di passare alla riceca sistematica degli iperparametri. <br>
L'early stopping è utile per prevenire l'overfitting.

In [ ]:
mlp1 = MLPClassifier(
    hidden_layer_sizes=(50,),
    activation="logistic",
    solver="adam",
    early_stopping=True,
    n_iter_no_change=40,
    max_iter=2000,
    validation_fraction=0.23,
    random_state=random_state,
)
mlp1.fit(X_trainval_pca, y_trainval)
y_pred_train_mlp1 = mlp1.predict(X_trainval_pca)
y_pred_test_mlp1  = mlp1.predict(X_test_pca)
y_proba_train_mlp1 = mlp1.predict_proba(X_trainval_pca)[:, 1]
y_proba_test_mlp1  = mlp1.predict_proba(X_test_pca)[:, 1]
auc_trainval_mlp1 = roc_auc_score(y_trainval, y_proba_train_mlp1)
auc_test_mlp1  = roc_auc_score(y_test,    y_proba_test_mlp1)
print(f"MLP #1: hidden=(50 neuroni), activation=logistic, dati PCA-ridotti")
print(f"  Epoche eseguite        : {len(mlp1.loss_curve_)}")
print(f"  Loss finale (training) : {mlp1.loss_curve_[-1]:.4f}")
print(f"  Accuracy training+val  : {mlp1.score(X_trainval_pca, y_trainval)*100:.2f}%")
print(f"  Accuracy test          : {mlp1.score(X_test_pca, y_test)*100:.2f}%")
print(f"  AUC training+val       : {auc_trainval_mlp1:.4f}")
print(f"  AUC test               : {auc_test_mlp1:.4f}")

Si ottengono valori di accuracy e di AUC accettabili, ma la loss sul training+validation set è piuttosto elevata.

Visualizzazione grafica delle learning curves di questa MLP:

In [ ]:
# Learning curve della MLP #1
fig = go.Figure()
fig.add_trace(go.Scatter(
    x=list(range(1, len(mlp1.loss_curve_)+1)),
    y=mlp1.loss_curve_,
    mode="lines", line=dict(color="#2E86AB", width=2), name="Training loss",
))
if hasattr(mlp1, "validation_scores_"):
    fig.add_trace(go.Scatter(
        x=list(range(1, len(mlp1.validation_scores_)+1)),
        y=mlp1.validation_scores_,
        mode="lines", line=dict(color="#E63946", width=2), name="Validation score",
        yaxis="y2",
    ))
fig.update_layout(
    title=f"Learning curve MLP #1 (1 layer × 50 neuroni, sigmoide)",
    xaxis_title="Epoca",
    yaxis=dict(title="Training loss"),
    yaxis2=dict(title="Validation accuracy", overlaying="y", side="right"),
    height=400, width=900, legend=dict(x=0.7, y=1.05),
)
fig.show()

L'MLP espolorativa con un singolo hidden layer e addestrata sui dati a dimensionalità ridotta ottiene un'AUC tra 0.88 e 0.89. La coerenza tra i valori dell'AUC (e dell'accuracy) tra il training+validation e il test set indica l'assenza di overfitting, ma la performance è modesta se confrontata con quella dell'SVM. Ciò suggerisce che una singola rete shallow non è sufficiente per catturare la complessità del problema e motiva la ricerca sistematica degli iperparametri che verrà affrontata nella sezione successiva.

### MLP & MAGIC Gamma Telescope: Grid Search per rete neurale profonda
Si effettua ora una grid search per individuare i parametri con i quali inizializzare il MLP con 5 layer. Essa esplora 3 architetture, 3 funzioni di attivazione e 2 batch size per un totale di 18 configurazioni, ciascuna addestrata sull'unione del training e del validation set e usando una pazienza (early stopping) di 75 epoche. Viene valutato l'AUC associato al test set. <br>
**Nota sulla concatenazione:** l'MLP con *early_stopping = True* usa*val_p*% campioni randomici dell'array passato in input come validation set interno per l'early stopping. <br>

In [ ]:
#lavorando con MLP il validation set è usato come tale
X_trainval_mlp = np.concatenate([X_train_scaled, X_val_scaled], axis = 0)
y_trainval_mlp = np.concatenate([y_train, y_val], axis = 0)
#iperparametri
hidden_layer_sizes_list = [[2**i]*5 for i in range(4,7)] #[16,32,64] neuroni x 5 layer 
activation_list = ['relu', 'logistic', 'tanh'] #3 possibili configurazioni
patience = 100
max_epochs = 1000
verbose = False #non stampo le info sullo stato di avanzamento
batch_sz_list = [2**i for i in range(5, 7)] #32, 64 (batch piccoli introducono più rumore nel gradiente)
val_p = 0.2 #% usata SOLO internamente come validation set per l'early stopping; X_val_scaled usato per la GRIDSEARCH sugli IPERPARAMETRI della rete
# in tutto ho 3*3*2 = 18 configurazioni da esplorare
results_mlp = []
for hidden_layer_sizes, activation, batch_sz in itertools.product(hidden_layer_sizes_list,
                                                                    activation_list, batch_sz_list):
    #inizializzazione MLP
    mlp_gs = MLPClassifier(hidden_layer_sizes = hidden_layer_sizes, activation = activation,
                    max_iter = max_epochs, n_iter_no_change=patience, early_stopping=True,
                    verbose=verbose, batch_size=batch_sz, validation_fraction = val_p,
                    random_state=random_state,
                    solver='adam')  #learning rate ignorato da solver Adam
    #early_stopping=True usa un train_test_split shuffled+stratificato per il val interno: l'ordine NON conta.
    mlp_gs.fit(X_train_scaled, y_train)
    auc = roc_auc_score(y_val, mlp_gs.predict_proba(X_val_scaled)[:, 1]) #utilizzo validation set per scegliere iperparametri
    results_mlp.append({
        'hidden_layer_sizes': hidden_layer_sizes,
        'batch_size':   batch_sz,
        'activation':        activation,
        'AUC_val':     auc,
        'n_iter':       mlp_gs.n_iter_
    })
results_mlp_df = pd.DataFrame(results_mlp).sort_values("AUC_val", ascending = False)
print("\nLe cinque migliori configurazioni sono:")
print(results_mlp_df.head(5).to_string(index=False))

Si osserva che la funzione tanh domina la classifica, suggerendo che un'attivazione liscia e limitata si adatta meglio alla struttura non lineare del problema di classificazione binaria. Inoltre, tanh è simmetrica. <br> Il modello finale fine riaddestrato con la configurazione che permette di raggiungere l'AUC più alta sul training set. 

In [ ]:
best_mlp           = results_mlp_df.iloc[0]
hidden_layer_sizes = best_mlp['hidden_layer_sizes']
batch_sz           = best_mlp['batch_size']
activation         = best_mlp['activation']
print(f"Hidden layer size ottimale: {hidden_layer_sizes}")
print(f"Batch size ottimale:        {batch_sz}")
print(f"Activation ottimale:        {activation}")

mlp = MLPClassifier(
    hidden_layer_sizes=hidden_layer_sizes, activation=activation,
    max_iter=max_epochs, n_iter_no_change=patience, early_stopping=True,
    verbose=False, batch_size=batch_sz, validation_fraction=val_p,
    random_state=random_state, solver='adam'  
)
#solver adam --> logloss (cross-entropy)
mlp.fit(X_trainval_mlp, y_trainval_mlp)  # train+val concatenati, anche se l'ordine di concatenazione è indifferente 
print(f"\nIterazioni effettive: {mlp.n_iter_}")
print(f"Loss finale:          {mlp.loss_:.6f}")

La loss misura la discrepanza tra le probabilità predette dalla rete e le etichette reali. Valori prossimi a zero indicano che il modello assegna probabilità elevate alla classe corretta. Nel caso in esame, una loss finale pari a circa 0.02 suggerisce che la MLP ha appreso una rappresentazione dei dati in grado di produrre predizioni generalmente accurate e caratterizzate da un elevato livello di confidenza.
Ciò conferma che un'architettura più profonda riesce ad apprendere una rappresentazione più ricca e complessa dei fatti. L'early stopping si attiva molto prima del massimo di 1000 epoche, indicando una convergenza stabile e senza necessità di addestramento prolungato

Le curve di addestramento mostrano l'accuracy sul validation set interno, usata da "MLPClassifier" per l'early stopping. <br> La valutazione finale del modello, però, viene condotta tramite l'AUC, più adatta a un dataset sbilanciato come quello in esame.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(mlp.loss_curve_, color='steelblue')
axes[0].set_xlabel("Epoca")
axes[0].set_ylabel("Loss")
axes[0].set_title("Training loss")
axes[0].grid(alpha=0.3)
axes[1].plot(mlp.validation_scores_, color='tomato')
axes[1].set_xlabel("Epoca")
axes[1].set_ylabel("Validation score (accuracy)")
axes[1].set_title("Validation score")
axes[1].grid(alpha=0.3)
plt.suptitle("MLP — Curve di addestramento", fontsize=13)
plt.tight_layout()
plt.show()

La curva di training loss decresce monotonicamente senza oscillazioni significative in una prima fase e, successivamente, inizia un raffinamento più lento e con leggere oscillazioni nelle epoche finali dovute alla natura stocastica del minibatch. La loss finale indica  una buona convergenza del modello. Il validation score (accuracy sul validation set interno) mostra un comportamento più irregolare e oscillatorio. La tendenza al ribasso nelle ultime epoche è un segnale di overfitting progressivo: il modello continua a migliorare sul training set, ma fatica a generalizzare sul validation set interno. In questa fase interviene l'early stopping, interrompendo l'addestramento dopo aver raggiunto la patience di 100 epoche senza miglioramento del validation score, evitando che il degeneri l'overfitting. 

Si indaga ora la performance della MLP e le matrici di confusione sul training+validation e sul test set.

In [ ]:
y_pred_trainval_mlp = mlp.predict(X_trainval_mlp)
y_proba_trainval_mlp = mlp.predict_proba(X_trainval_mlp)[:,1]
y_pred_mlp = mlp.predict(X_test_scaled)
y_proba_mlp = mlp.predict_proba(X_test_scaled)[:,1]
acc_trainval= mlp.score(X_trainval_mlp, y_trainval_mlp)
prec_trainval = precision_score(y_trainval_mlp, y_pred_trainval_mlp, average='weighted')
rec_trainval = recall_score(y_trainval_mlp, y_pred_trainval_mlp, average='weighted')
auc_trainval = roc_auc_score(y_trainval_mlp, y_proba_trainval_mlp)
acc = mlp.score(X_test_scaled, y_test)
prec = precision_score(y_test, y_pred_mlp, average = "weighted") #occorre tenere conto dello sbilanciamento
rec = recall_score(y_test, y_pred_mlp, average = "weighted") #occorre tenere conto dello sbilanciamento
auc = roc_auc_score(y_test, y_proba_mlp) #classificatore binario, non necessita di weighted perchè la metrica è già intrinsecamente robusta allo sbilanciamento
df_perf_mlp = pd.DataFrame({'Accuracy': [acc_trainval, acc], 
                        'Precision': [prec_trainval, prec], 
                        'Recall': [rec_trainval, rec],
                        'AUC': [auc_trainval, auc]
                       },
                      index=['train. + val.', 'test'])
display(df_perf_mlp.round(4))
#matrici di confusione
cmat_mlp           = confusion_matrix(y_test, y_pred_mlp, labels=mlp.classes_)
cmat_mlp_norm_true = confusion_matrix(y_test, y_pred_mlp, labels=mlp.classes_, normalize='true')
cmat_mlp_norm_pred = confusion_matrix(y_test, y_pred_mlp, labels=mlp.classes_, normalize='pred')
df_cmat_mlp           = pd.DataFrame(cmat_mlp, columns=class_names, index=class_names)
df_cmat_mlp_norm_true = pd.DataFrame(cmat_mlp_norm_true, columns=class_names, index=class_names)
df_cmat_mlp_norm_pred = pd.DataFrame(cmat_mlp_norm_pred, columns=class_names, index=class_names)
print("\nMatrice di confusione (valori assoluti):")
display(df_cmat_mlp)
print("\nMatrice di confusione (normalizzata rispetto alle vere classi — recall):")
display(df_cmat_mlp_norm_true.round(4))
print("\nMatrice di confusione (normalizzata rispetto alle classi predette — precision):")
display(df_cmat_mlp_norm_pred.round(4))
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
ConfusionMatrixDisplay(cmat_mlp,
    display_labels=class_names).plot(
    ax=axes[0], colorbar=False, cmap='Blues')
axes[0].set_title("Valori assoluti")
ConfusionMatrixDisplay(cmat_mlp_norm_true,
    display_labels=class_names).plot(
    ax=axes[1], colorbar=False, cmap='Blues', values_format='.3f')
axes[1].set_title("Normalizzata su vere classi (recall)")
ConfusionMatrixDisplay(cmat_mlp_norm_pred,
    display_labels=class_names).plot(
    ax=axes[2], colorbar=False, cmap='Blues', values_format='.3f')
axes[2].set_title("Normalizzata su classi predette (precision)")
plt.suptitle("Matrici di confusione — MLP (test set)", fontsize=13)
plt.tight_layout()
plt.show()

La tabella delle performance mostra un comportamento complessivamente coerente tra i due set (trainval e test set), mostrando un gap contenuto e fisiologico. 

## Classificazione - curve ROC e conclusioni

Visualizzazione delle curve ROC dei metodi di classificazione:

In [ ]:
y_proba_mlp = mlp.predict_proba(X_test_scaled)[:,1]
fpr_mlp, tpr_mlp, _ = roc_curve(y_test, mlp.predict_proba(X_test_scaled)[:, 0], pos_label=0)
fpr_svm, tpr_svm, _ = roc_curve(y_test, svm_best.predict_proba(X_test_scaled_tv)[:, 0], pos_label=0)
fpr_qda, tpr_qda, _ = roc_curve(y_test, qda.predict_proba(X_test_scaled)[:, 0], pos_label=0)

plt.figure(figsize=(7, 6))
plt.plot(fpr_mlp, tpr_mlp,
         label=f"MLP  (AUC = {auc:.4f})", color='seagreen')
plt.plot(fpr_svm, tpr_svm,
         label=f"SVM RBF  (AUC = {auc_svm_test:.4f})", color='steelblue',
         linestyle='--')
plt.plot(fpr_qda, tpr_qda,
         label=f"QDA  (AUC = {auc_qda_t:.4f})", color='tomato',
         linestyle='--')
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("Curva ROC — MLP vs SVM vs QDA (test set)")
plt.legend()
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

Stampiamo una tabella riassuntiva.

In [ ]:
summary = pd.DataFrame({
    'Modello':  ['QDA', 'SVM RBF', 'MLP'],
    'AUC Test': [auc_lda_test, auc_qda_t, auc_svm_test, auc],
    'Precision': [
        
        precision_score(y_test.astype(int), qda.predict(X_test_scaled),     average='weighted'),
        precision_score(y_test.astype(int), y_test_pred_svm,                average='weighted'),
        precision_score(y_test,             y_pred_mlp,                     average='weighted')
    ],
    'Recall': [
        
        recall_score(y_test.astype(int), qda.predict(X_test_scaled),     average='weighted'),
        recall_score(y_test.astype(int), y_test_pred_svm,                average='weighted'),
        recall_score(y_test,             y_pred_mlp,                     average='weighted')
    ],
})
print("\n=== RIEPILOGO FINALE sul test set ===")
print(summary.to_string(index=False))

Come indicato nella documentazione del dataset, l'accuracy non è una 
metrica adeguata per questo problema: classificare un evento adronico come 
segnale gamma (falso positivo) è un errore più costoso rispetto al caso opposto, 
poiché inquina il campione di eventi accettati con fondo non desiderato. 
La metrica di riferimento è quindi la curva ROC, e i punti rilevanti sono quelli 
in cui il false positive rate (probabilità di accettare un evento adronico come 
segnale) è al di sotto delle soglie operative tipiche degli esperimenti: 
0.01, 0.02, 0.05, 0.1 e 0.2. La tabella seguente riporta, per ciascuna soglia 
e per i tre modelli migliori, il true positive rate massimo raggiungibile — 
ovvero la frazione di eventi gamma correttamente identificati mantenendo il 
fondo entro il limite fissato.

In [ ]:
thresholds_fpr = [0.01, 0.02, 0.05, 0.1, 0.2]
gamma_curves = {
    'SVM RBF': roc_curve(y_test, svm_best.predict_proba(X_test_scaled_tv)[:, 0], pos_label=0),
    'MLP':     roc_curve(y_test, mlp.predict_proba(X_test_scaled)[:, 0],      pos_label=0),
    'QDA':     roc_curve(y_test, qda.predict_proba(X_test_scaled)[:, 0],      pos_label=0),
}
rows = []
for fpr_th in thresholds_fpr:
    row = {'FPR <= (hadron->gamma)': fpr_th}
    for name, (fpr, tpr, _) in gamma_curves.items():
        mask = fpr <= fpr_th
        row[name] = tpr[mask].max() if mask.any() else 0.0   # max TPR (recall gamma) con FPR sotto soglia
    rows.append(row)
df_roc = pd.DataFrame(rows).set_index('FPR <= (hadron->gamma)')
display(df_roc.round(4))

## Discussione e Conclusioni
### Riepilogo numerico




### 1. PCA come strumento esplorativo
Sul dataset MAGIC la PCA non costituisce uno strumento di riduzione della 
dimensionalità efficace: sono necessarie 7 componenti su 10 per raggiungere 
il 95% di varianza spiegata, a indicare una ridondanza informativa limitata 
tra le feature originali. Il suo contributo è principalmente esplorativo: 
la PC1 si rivela un indicatore di intensità globale del segnale Cherenkov, 
mentre PC2 e PC3 catturano asimmetrie morfologiche dell'ellisse. La 
performance sostanzialmente invariata di SVM e MLP con e senza riduzione 
dimensionale conferma che la PCA non aggiunge potere discriminativo in 
questo contesto.

### 2. FDA come strumento esporativo
La FDA costituisce per un problema binario la controparte geometrica dell'LDA. Tale equivalenza è confermata dalla fortissima correlazione tra i coefficienti normalizzati dei vettori discriminanti identificati da questi metodi. Le proiezioni mododimensionali, inoltre, mostrano che l'FDA separa maggiormente le due classi rispetto alla PCA, a conferma del vantaggio dell'approccio supervisionato. Come la PCA, la FDA viene utilizzata solo come strumento esplorativo: la riduzione a una sola dimensione non è sufficiente per un problema non linearmente separabile.

### 3. Classificatori
I quattro classificatori esaminati mostrano prestazioni crescenti in termini
di AUC sul test set: LDA (0.841) < QDA (0.871) < SVM RBF (0.924) < MLP (0.931).
 
**LDA**: si tratta di un modello lineare che
presuppone gaussianità e omoschedasticità tra le classi, ipotesi
che l'analisi esplorativa ha mostrato essere violate (le matrici di covarianza
di gamma e hadron differiscono significativamente in norma di Frobenius).
Questo spiega il suo AUC più basso.
 
**QDA**: si rilassa l'ipotesi di omoschedasticità stimando una matrice di varianza e covarianza separata
per ogni classe, guadagnando 3 punti di AUC rispetto a LDA (0.871 vs 0.841).
Il miglioramento è modesto ma consistente, confermando che la struttura di
covarianza differente tra le classi porta informazione discriminante reale.
 
**SVM con kernel RBF**: porta un salto netto di AUC a 0.924 (+5.3 punti rispetto a
QDA). La capacità di apprendere frontiere di decisione non lineari nello
spazio delle feature originali, senza alcuna assunzione distribuzionale,
si rivela decisiva su questo dataset. La regolarizzazione tramite il parametro
C selezionato con grid search previene l'overfitting nonostante la maggiore
flessibilità del modello.
 
**MLP** (64×5 neuroni, tanh, adam): raggiunge l'AUC più alto (0.931), superando
l'SVM RBF di quasi 1 punto. La rete a cinque strati nascosti riesce a catturare
interazioni tra feature di ordine superiore che il kernel RBF approssima meno
efficacemente. Il vantaggio è contenuto, il che suggerisce che la complessità
aggiuntiva dell'MLP porta un guadagno marginale rispetto all'SVM in questo
contesto.
 
### Analisi ai punti operativi
 
Il contesto fisico del telescopio MAGIC impone un vincolo stringente sul
**False Positive Rate**: classificare erroneamente un evento adronico come
segnale gamma (FPR alto) inquina il campione scientifico e rende i risultati
inaffidabili. Per questo motivo il confronto ai punti operativi (FPR ≤ 0.01
e FPR ≤ 0.05) è più informativo della sola AUC.
 
A **FPR ≤ 0.01** (contaminazione adronica massima dell'1%):
- MLP recupera il 28.6% dei fotoni gamma reali
- QDA recupera il 24.6%
- SVM RBF recupera solo il 19.1%
 
A **FPR ≤ 0.05** (contaminazione massima del 5%):
- MLP recupera il 59.7% dei fotoni gamma
- SVM RBF recupera il 53.9%
- QDA recupera il 49.7%
 
L'MLP domina in entrambi i punti operativi rilevanti: non solo ha l'AUC
globale più alta, ma mantiene il vantaggio anche nelle regioni di basso FPR
dove il telescopio opera realisticamente. Il vantaggio dell'MLP sull'SVM
è più marcato (circa 6 punti percentuali a FPR ≤ 0.05)
di quanto non lo sia in AUC globale (~1 punto), il che indica che l'MLP
è particolarmente efficace nel ridurre i falsi positivi.
 
### Raccomandazione finale
 
L'**MLP [64,64,64,64,64] con attivazione tanh** è il modello raccomandato
per la classificazione degli eventi del telescopio MAGIC. Rispetto all'SVM
RBF, offre un TPR superiore di circa 6 punti percentuali alla soglia
operativa FPR ≤ 0.05, mantenendo il medesimo livello di purezza del campione.
Il costo computazionale aggiuntivo dell'MLP rispetto all'SVM è giustificato
da questo guadagno nelle condizioni di utilizzo reale dello strumento.



# Appendice: Metriche

## L'Accuracy, la Precision, il Recall (o Sensibilità)

L'*accuracy*  misura la frazione di predizioni corrette rispetto al
totale dei campioni:
$$
\text{Accuracy} = \frac{TP + TN}{TP + TN + FP + FN},
$$ 
dove dove $TP$
indica il numero di esempi positivi classificati correttamente, $FN$ il
numero di falsi negativi, $FP$ il numero di falsi positivi e $TN$ il
numero di esempi negativi classificati correttamente.\
L'accuracy fornisce una valutazione globale del modello, ma, nel caso di
dataset sbilanciati o di costi asimmetrici degli errori, risulta poco
informativa.\
La *precision* misura l'affidabilità delle predizioni positive:
$$
\text{precision} = \frac{TP}{TP + FP},
$$ 
ossia la proporzione di veri
positivi tra tutti gli esempi classificati come tali.\
Il *recall* (o *sensibilità* o *true positive rate*) quantifica la capaacità del modello di
individuare correttamente le istanze positive:
$$
\text{recall} = TPR = \frac{TP}{TP + FN}.
$$ 
Si osserva, dunque, che
precision e recall descrivono due aspetti complementari del
comportamento del classificatore: la prima penalizza i falsi positivi,
mentre la seconda penalizza i falsi negativi.\

## La Curva ROC (Receiver Operating Characteristic)

La *receiver operating characteristic (ROC)* è una curva che descrive le
prestazioni di un classificatore binario al variare del valore di una
soglia di decisione $\tau \in [0, 1]$ (considerando anche i casi
limite). Come l'AUC, tiene conto simultaneamente del trade-off tra falsi
positivi e falsi negativi al variare della soglia di decisione.\
Dato un sample $x$ e un classificatore che produce la probabilità
$\tilde{p}(x)$ per la classe positiva, la predizione è
$$
\begin{equation*}
    y = 
    \begin{cases}
        1, \text{se } \tilde{p}(x) \geq \tau \\
        0, \text{altrimenti}
    \end{cases}
\end{equation*}
$$ 
Per ogni valore di $\tau$, è possibile valutare il
*true positive rate (TPR, recall*, la formula è la stessa, ma si ha una dipendenza di TP, FN e TPR stesso da $\tau$)* e il *false positive rate (FPR)*, anche noto come
*1-specificità*: 
$$
\begin{equation*}
    FPR(\tau) = \frac{FP(\tau)}{FP(\tau) + TN(\tau)} = P(\tilde{p}(x) \geq \tau | y = 0),
\end{equation*}
$$ 
dove $TP(\tau)$ indica il numero di esempi positivi
classificati correttamente, $FN(\tau)$ il numero di falsi negativi,
$FP(\tau)$ il numero di falsi positivi e $TN(\tau)$ il numero di esempi
negativi classificati correttamente per un valore fissato di
$\tau \in [0,1]$.\
La curva ROC traccia al variare di $\tau \in [0,1]$ il punto
$(FPR(\tau), TPR(\tau))$.\
Si osserva che l'origine corrisponde al punto $\tau = 1$, ovvero al caso
degenere in cui nessun campione viene classificato come positivo.
Analogamente, il punto $(1,1)$ rappresenta il caso $\tau = 0$, in cui
tutti i campioni sono classificati come positivi. Inoltre, la diagonale
$TPR = FPR$ rappresenta un classificatore casuale, ovvero tale per cui
la sua AUC valga $0.5$. Un classificatore perfetto raggiunge una AUC
pari a $1.0$ passando per il punto $(0,1)$.

## L'AUC (Area Under the Curve)

L'*area sottesa dalla curva ROC* (in inglese *area under the curve,
AUC*) si ottiene tramite la formula: 
$$
\begin{equation*}
    AUC = \int_{0}^{1} TPR(FPR) \,d(FPR)
\end{equation*}
$$ 
e assume valori in $[0,1]$. L'AUC rappresenta la
probabilità che il classificatore assegni un \"punteggio\" più alto a un
campione positivo ($x^+$) rispetto a uno negativo ($x^-$):
$$
\begin{equation*}
    AUC = P(\tilde{p}(x^+) > \tilde{p}(x^-)).
\end{equation*}
$$
L'AUC è, dunque, una misura discriminante indipendente
dalla soglia e si rivela essere molto utile per confrontare modelli
diversi.\
Si ritiene accettabile un classificatore con AUC compresa tra 0.7 e 0.8,
buono tra 0.8 e 0.9 e ottimo per valori superiori. Tuttavia, valori
elevati (\>0.95) di AUC possono essere segnale di overfitting.

## L'AUC e il Dataset MAGIC Gamma Telescope

La documentazione ufficiale del dataset \"MAGIC Gamma Telescope\" specifica esplicitamente che l'accuracy
non è una metrica significativa per questo problema di classificazione
binaria. Ciò è dovuto allo sbilanciamento delle classi (*gamma* 64.8 %,
*hadron* 35.2%): un classificatore che predice
sempre gamma ottiene un'accuracy pari a $0.648$ e non apprende alcuna
struttura intrinseca ai dati. Nella scelta della metrica AUC influisce
anche la natura stessa del problema fisico: il costo di classificare
erroneamente un evento adronico come segnale gamma (falso positivo) non
è uguale al costo dell'errore opposto (falso negativo).\
L'area sottesa dalla curva ROC è capace di catturare il trade-off tra i
due tipi di errore e su tutte le soglie possibili, fornendo una
valutazione globale e comparabile tra i modelli.

## Il F1-Score

Un'altra metrica ampiamente utilizzata è il *F1-score* che rappresenta
la media armonica tra precision e recall:
$$
F1 = 2 \cdot \frac{\text{Precision} \cdot \text{Recall}}{\text {Precision} + \text{Recall}}.
$$
Così facendo, il F1-score fornisce una misura unica del bilanciamento
tra le due metriche, risultando particolarmente utile in presenza di
dataset sbilanciati. A differenza dell'accuracy, infatti, esso si
concentra esclusivamente sulla qualità delle predizioni positive.\
Precision e recall sono inoltre legate da un tipico *trade-off*:
aumentando la soglia di decision $\tau$, il modello tende ad essere più
conservativo, incrementando la precision, ma riducendo il recall;
viceversa, diminuendo $\tau$, il recall crese a scapito della precision.
Tale compromesso può essere rappresentato tramite la *Precision-Recall
curve* che descrive l'andamento della precision al variare del recall per
tutte le possibili soglie di classificazione.